In [ ]:
# =============================================================================
# Zelle 01 – Setup Phase 4 (Notebook 05: Matching)
# =============================================================================
# Zweck: Gemeinsame Pfade, Konfiguration und Speicherfunktion für Phase 4
#   (Fuzzy, Vetos, Inhaltsbelege, ML-Paar-Klassifikator).
# Begründung: Baseline A (NB04) bleibt unverändert; NB05 lädt deren Ergebnisse
#   und misst nur den Rest, den der deterministische Kern nicht auflöst.
# Datenschutz (P11): echte Werte nur in data/ und reports/errors/.
# =============================================================================

from functools import partial
from pathlib import Path

import pandas as pd

from playlist_agent import projekt
from playlist_agent.viz_config import stil_aktivieren

# Zentraler Kontext (Arbeitspaket 4.8): Wurzel über pyproject.toml, cfg, Pfade
K = projekt.kontext()
PROJECT_ROOT, cfg, SEED = K.wurzel, K.cfg, K.seed
RAW, INTERIM, GOLD = K.raw, K.interim, K.gold
FIG, TAB, ERR = K.fig, K.tab, K.err
stil_aktivieren()

# Speichern ohne stilles Überschreiben (gleicher Inhalt: ok, anderer: Fehler)
tabelle_speichern = partial(projekt.tabelle_speichern, ordner=TAB)


# Kontrolle: alle Ordner vorhanden
for _n, _p in {"RAW": RAW, "INTERIM": INTERIM, "GOLD": GOLD,
               "FIG": FIG, "TAB": TAB, "ERR": ERR}.items():
    print(f"{_n:8s} {'vorhanden' if _p.is_dir() else 'FEHLT'}")

In [ ]:
# =============================================================================
# Zelle 02 – Inventar der NB04-Artefakte
# =============================================================================
# Zweck: Feststellen, welche Ergebnisse aus Notebook 04 (Präfix 04_) in
#   data/interim, data/gold und reports/tables vorliegen und welche NB05
#   laden kann, statt neu zu rechnen.
# Begründung: Dateinamen werden gemessen, nicht angenommen.
# =============================================================================

zeilen = []
for _name, _ordner in {"interim": INTERIM, "gold": GOLD, "tables": TAB}.items():
    for _p in sorted(_ordner.glob("04_*")):
        zeilen.append({"ordner": _name, "datei": _p.name,
                       "groesse [KB]": round(_p.stat().st_size / 1024, 1)})

df_inventar = pd.DataFrame(zeilen)
print(f"{len(df_inventar)} Dateien mit Präfix 04_")
print(df_inventar.to_string(index=False))
_ = tabelle_speichern(df_inventar, "05_02_inventar_nb04")

In [ ]:
# =============================================================================
# Zelle 03 – Struktur der NB04-Kernergebnisse (Index, Resolver Crates/Playlists)
# =============================================================================
# Zweck: Spalten und Umfang der Dateien feststellen, die NB05 lädt.
# Begründung: Spaltennamen werden gemessen, nicht angenommen. Es werden keine
#   Zeilen ausgegeben, da sie echte Pfade enthalten (P11).
# =============================================================================

DATEIEN = {"index": INTERIM / "04_04_index_aktuell.csv",
           "res_crates": INTERIM / "04_05_resolver_crates.csv",
           "res_playlists": INTERIM / "04_05_resolver_playlists.csv"}

for _name, _pfad in DATEIEN.items():
    _df = pd.read_csv(_pfad, low_memory=False)
    print(f"--- {_name}: {len(_df)} Zeilen, {_df.shape[1]} Spalten")
    print(_df.dtypes.to_string())
    print()

In [ ]:
# =============================================================================
# Zelle 04 – Ergebnisklassen und Stufen (NB04) als Ausgangsbasis für Phase 4
# =============================================================================
# Zweck: Zählung je Klasse/Stufe prüfen und gegen die NB04-Kennzahlen abgleichen.
# Begründung: NB05 baut nur auf Zahlen auf, die reproduziert wurden.
# Erwartung res_playlists: sicher 8.493 | stream 2 | vorschlag 1.026 | fehlt 71
#   | album_beschaedigt 678 | ausgeschlossen 19 (Summe 10.289).
# Erwartung res_crates: fehlt 8 Pfade.
# Datenschutz: nur Zählwerte, keine Pfade.
# =============================================================================

df_rc = pd.read_csv(DATEIEN["res_crates"], low_memory=False)
df_rp = pd.read_csv(DATEIEN["res_playlists"], low_memory=False)

df_klassen = pd.concat([
    df_rc.groupby(["klasse", "stufe"]).size().rename("anzahl [Pfade]").reset_index().assign(quelle="crates"),
    df_rp.groupby(["klasse", "stufe"]).size().rename("anzahl [Eintraege]").reset_index().assign(quelle="playlists"),
])
print(df_klassen.to_string(index=False))
print("\nKlassen gesamt:")
print("crates   :", df_rc["klasse"].value_counts().to_dict(), f"(Summe {len(df_rc)})")
print("playlists:", df_rp["klasse"].value_counts().to_dict(), f"(Summe {len(df_rp)})")
_ = tabelle_speichern(df_klassen, "05_04_klassen_stufen_nb04")

In [ ]:
# =============================================================================
# Zelle 05 – "fehlt": existiert der Ordner auf dem Stick? (Kontrolle: sicher/pfad)
# =============================================================================
# Zweck: Die 8 Crate-Pfade und 71 Playlist-Einträge ohne Kandidaten einordnen:
#   a) Ordner vorhanden, Datei nicht (Datei umbenannt / Zeichen verloren)
#   b) Ordner nicht vorhanden (Ordner umbenannt / verschoben / gelöscht)
# Begründung: Entscheidet, ob Fuzzy innerhalb eines Ordners (kleiner Suchraum)
#   genügt oder ein Ordner-Matching nötig ist.
# Kontrolle: bei "sicher/pfad" muss der Ordner zu 100 % existieren, sonst ist
#   der Vergleich (NFC, Trenner) ungültig.
# Datenschutz: Konsole nur Zählwerte; Einzelfälle privat in reports/errors.
# =============================================================================

import posixpath

from playlist_agent.canon import nfc

df_ix = pd.read_csv(DATEIEN["index"], low_memory=False)
_pfade = df_ix["pfad_relativ"].map(lambda s: nfc(str(s)).replace("\\", "/"))
ORDNER_IDX = set()
for _p in _pfade:
    _d = posixpath.dirname(_p)
    while _d and _d not in ORDNER_IDX:      # alle Vorfahren aufnehmen
        ORDNER_IDX.add(_d)
        _d = posixpath.dirname(_d)


def ordner_status(pfad: str) -> dict:
    """Ordner vorhanden? Wie viele Pfadteile existieren von oben her?"""
    p = nfc(str(pfad)).replace("\\", "/")
    teile = p.split("/")[:-1]               # ohne Dateiname
    vorhanden = 0
    for i in range(1, len(teile) + 1):
        if "/".join(teile[:i]) in ORDNER_IDX:
            vorhanden = i
        else:
            break
    return {"ordner_ok": vorhanden == len(teile),
            "teile_gesamt": len(teile), "teile_vorhanden": vorhanden}


def pruefe(df: pd.DataFrame, spalte: str, klasse: str, stufe: str | None, quelle: str):
    sel = df[(df["klasse"] == klasse) & ((df["stufe"] == stufe) if stufe else True)]
    pfade = sorted(set(sel[spalte].astype(str)))
    out = pd.DataFrame([{"pfad": p, **ordner_status(p)} for p in pfade])
    out["quelle"], out["gruppe"] = quelle, f"{klasse}/{stufe or '*'}"
    return out


df_ordner = pd.concat([
    pruefe(df_rc, "eintrag", "fehlt", None, "crates"),
    pruefe(df_rp, "kanonisch", "fehlt", None, "playlists"),
    pruefe(df_rc, "eintrag", "sicher", "pfad", "crates"),        # Kontrolle
    pruefe(df_rp, "kanonisch", "sicher", "pfad", "playlists"),   # Kontrolle
], ignore_index=True)

df_z05 = (df_ordner.groupby(["quelle", "gruppe"])
          .agg(pfade=("pfad", "size"), ordner_ok=("ordner_ok", "sum"))
          .reset_index().rename(columns={"pfade": "pfade [Pfade]",
                                         "ordner_ok": "ordner_vorhanden [Pfade]"}))
print(df_z05.to_string(index=False))
print("\nFehlt: Pfadteile vorhanden / gesamt (nur Ordner nicht vorhanden):")
_f = df_ordner[(df_ordner["gruppe"] == "fehlt/*") & ~df_ordner["ordner_ok"]]
print(_f.groupby(["quelle", "teile_vorhanden", "teile_gesamt"]).size()
      .rename("pfade [Pfade]").to_string())
_ = tabelle_speichern(df_z05, "05_05_fehlt_ordnerstatus")
df_ordner[df_ordner["gruppe"] == "fehlt/*"].to_csv(
    ERR / "05_05_fehlt_ordnerstatus_faelle.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 06 – "fehlt", Ordner nicht vorhanden: gleichnamiger Ordner anderswo?
# =============================================================================
# Zweck: Für die 24 Playlist-Pfade ohne vorhandenen Ordner prüfen, ob der
#   unmittelbare Elternordner (Basisname) im Index als Ordner existiert
#   (exakt NFC / ohne Gross-Kleinschreibung) und ob die oberste Komponente
#   dem Muster 'Musik/...' folgt.
# Begründung: Trennt Präfix-Problem (deterministisch lösbar) von echtem
#   Ordner-Verlust oder -Umbenennung (Phase-4-Kandidat für Fuzzy/ML).
# Datenschutz: Konsole nur Zählwerte; Einzelfälle privat in reports/errors.
# =============================================================================

from collections import defaultdict

BASENAME_EXAKT = defaultdict(set)
BASENAME_FOLD = defaultdict(set)
for _d in ORDNER_IDX:
    _b = posixpath.basename(_d)
    BASENAME_EXAKT[_b].add(_d)
    BASENAME_FOLD[_b.casefold()].add(_d)

_k = df_ordner[(df_ordner["gruppe"] == "fehlt/*") & ~df_ordner["ordner_ok"]
               & (df_ordner["quelle"] == "playlists")].copy()
_k["norm"] = _k["pfad"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_k["beginnt_musik"] = _k["norm"].str.startswith("Musik/")
_k["eltern"] = _k["norm"].map(lambda s: posixpath.basename(posixpath.dirname(s)))
_k["n_exakt"] = _k["eltern"].map(lambda b: len(BASENAME_EXAKT.get(b, ())))
_k["n_fold"] = _k["eltern"].map(lambda b: len(BASENAME_FOLD.get(b.casefold(), ())))

df_z06 = (_k.groupby(["beginnt_musik", "teile_vorhanden", "n_exakt", "n_fold"])
          .size().rename("pfade [Pfade]").reset_index())
print(f"{len(_k)} Pfade ohne vorhandenen Ordner")
print(df_z06.to_string(index=False))
_ = tabelle_speichern(df_z06, "05_06_ordner_fehlt_basisname")
_k.drop(columns="norm").to_csv(ERR / "05_06_ordner_fehlt_basisname_faelle.csv",
                               index=False, encoding="utf-8")        # privat

In [ ]:
# =============================================================================
# Zelle 07 – Alle ungelösten Fälle im Klartext (Sichtprüfung durch den Nutzer)
# =============================================================================
# Zweck: Die Pfade aus "fehlt" (Crates 8, Playlists 52) und "mehrfach" (2) zur
#   Prüfung durch den Nutzer anzeigen; Gruppe wie in Zelle 05/06.
# Begründung: Gold-Labels entstehen aus Nutzerprüfung (Explorer/foobar); der
#   Nutzer kennt seine Sammlung. Nur lokale Anzeige; Datei privat (P11).
# Gruppen: A Ordner vorhanden, Datei fehlt | B Ordner fehlt, Basisname 1x
#   anderswo | C Ordner fehlt, kein gleichnamiger Ordner, kein Musik-Präfix
#   | D Ordner fehlt, Präfix Musik/ | M mehrfach
# =============================================================================

pd.set_option("display.max_colwidth", None, "display.width", 250)

_z = df_ordner[df_ordner["gruppe"] == "fehlt/*"].copy()
_z["norm"] = _z["pfad"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_z["eltern"] = _z["norm"].map(lambda s: posixpath.basename(posixpath.dirname(s)))
_z["n_exakt"] = _z["eltern"].map(lambda b: len(BASENAME_EXAKT.get(b, ())))


def gruppe(r) -> str:
    if r["ordner_ok"]:
        return "A"
    if r["n_exakt"] >= 1:
        return "B"
    return "D" if r["norm"].startswith("Musik/") else "C"


_z["fall"] = _z.apply(gruppe, axis=1)
df_faelle = _z[["fall", "quelle", "pfad"]].sort_values(["fall", "quelle", "pfad"])

_m = df_rp[df_rp["stufe"] == "mehrfach"][["kanonisch", "n_kand", "pfad_ziel"]]
_m = _m.rename(columns={"kanonisch": "pfad"}).assign(fall="M", quelle="playlists")
df_faelle = pd.concat([df_faelle, _m[["fall", "quelle", "pfad", "n_kand", "pfad_ziel"]]],
                      ignore_index=True)

print(df_faelle["fall"].value_counts().sort_index().to_string(), "\n")
print(df_faelle.to_string(index=False))
df_faelle.to_csv(ERR / "05_07_ungeloeste_faelle.csv", index=False, encoding="utf-8")  # privat

In [ ]:
# =============================================================================
# Zelle 08 – Gruppe A: Dateien im vorhandenen Ordner und ähnlichster Name
# =============================================================================
# Zweck: Für die 36 Pfade (Ordner vorhanden, Datei fehlt) messen, ob im
#   Ordner überhaupt Dateien liegen und wie ähnlich der ähnlichste Name ist
#   (difflib.SequenceMatcher, Basisname ohne Endung, casefold).
# Begründung: Erst messen, ob Fuzzy-Matching überhaupt Kandidaten hat, bevor
#   Schwellen und Vetos festgelegt werden. Kein Urteil über Richtigkeit: das
#   entscheidet die Nutzerprüfung (Gold).
# Datenschutz: Konsole nur Zählwerte; Einzelfälle privat in reports/errors.
# =============================================================================

from difflib import SequenceMatcher

_dateien_je_ordner = defaultdict(list)
for _p in _pfade:
    _dateien_je_ordner[posixpath.dirname(_p)].append(posixpath.basename(_p))


def _stamm(name: str) -> str:
    return posixpath.splitext(name)[0].casefold()


_a = df_faelle[df_faelle["fall"] == "A"].copy()
_a["norm"] = _a["pfad"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_zeilen = []
for _, r in _a.iterrows():
    ordner = posixpath.dirname(r["norm"])
    kand = _dateien_je_ordner.get(ordner, [])
    best, score = "", 0.0
    for k in kand:
        s = SequenceMatcher(None, _stamm(posixpath.basename(r["norm"])), _stamm(k)).ratio()
        if s > score:
            best, score = k, s
    _zeilen.append({"quelle": r["quelle"], "pfad": r["norm"],
                    "dateien_im_ordner [Dateien]": len(kand),
                    "bester_kandidat": best, "aehnlichkeit": round(score, 3)})

df_a = pd.DataFrame(_zeilen)
print(f"{len(df_a)} Pfade; Ordner ohne Dateien: {(df_a['dateien_im_ordner [Dateien]'] == 0).sum()}")
print(df_a["aehnlichkeit"].describe().round(3).to_string())
print(pd.cut(df_a["aehnlichkeit"], [-0.01, 0.5, 0.7, 0.8, 0.9, 1.0]).value_counts()
      .sort_index().rename("pfade [Pfade]").to_string())
df_a.to_csv(ERR / "05_08_gruppe_a_kandidaten.csv", index=False, encoding="utf-8")   # privat
_ = tabelle_speichern(df_a[["quelle", "dateien_im_ordner [Dateien]", "aehnlichkeit"]],
                      "05_08_gruppe_a_aehnlichkeit")

In [ ]:
# =============================================================================
# Zelle 09 – Gruppe A: Ziffern-Veto auf den besten Namenskandidaten
# =============================================================================
# Zweck: Prüfen, ob der ähnlichste Name im Ordner dieselben Zahlenfolgen
#   (Tracknummer, Datum) enthält wie der gesuchte Name. Unterschiedliche
#   Zahlen bei hoher Ähnlichkeit deuten auf einen Nachbartitel hin.
# Begründung: Veto-Kandidat für Phase 4; Wirkung wird gemessen, nicht
#   angenommen. Kein Urteil über Richtigkeit: das entscheidet die Gold-Prüfung.
# Datenschutz: Zählwerte in tables/; Einzelfälle nur lokale Anzeige und privat.
# =============================================================================

import re

_ZIFFERN = re.compile(r"\d+")


def ziffern(name: str) -> tuple:
    return tuple(_ZIFFERN.findall(posixpath.splitext(name)[0]))


df_a["name_gesucht"] = df_a["pfad"].map(posixpath.basename)
df_a["ziffern_gleich"] = [
    bool(k) and ziffern(g) == ziffern(k)
    for g, k in zip(df_a["name_gesucht"], df_a["bester_kandidat"])]
df_a["klasse_aehnl"] = pd.cut(df_a["aehnlichkeit"], [-0.01, 0.5, 0.7, 0.8, 0.9, 1.0])

df_z09 = (df_a.groupby(["klasse_aehnl", "ziffern_gleich"], observed=True).size()
          .rename("pfade [Pfade]").reset_index())
print(df_z09.to_string(index=False))

print("\nOrdner ohne Dateien:")
print(df_a.loc[df_a["dateien_im_ordner [Dateien]"] == 0, "pfad"].to_string(index=False))

print("\nFälle ab Ähnlichkeit 0,8 (zur Sichtprüfung):")
_s = df_a[df_a["aehnlichkeit"] >= 0.8].sort_values("aehnlichkeit")
print(_s[["name_gesucht", "bester_kandidat", "aehnlichkeit", "ziffern_gleich"]]
      .to_string(index=False))

_ = tabelle_speichern(df_z09.assign(klasse_aehnl=df_z09["klasse_aehnl"].astype(str)),
                      "05_09_ziffern_veto")
df_a.drop(columns="klasse_aehnl").to_csv(ERR / "05_09_gruppe_a_ziffern.csv",
                                         index=False, encoding="utf-8")    # privat

In [ ]:
# =============================================================================
# Zelle 10 – Nutzerziele zu den 62 ungelösten Fällen laden (Gold-Datei)
# =============================================================================
# Zweck: Die handschriftlichen Ziele (Spalte 6) und Kommentare (Spalte 7) aus
#   data/gold/05_07_ungeloeste_labels.txt einlesen und je Fall klassifizieren:
#   datei (Audio-Endung) | ordner (Laufwerk, keine Audio-Endung) | text
#   (z. B. "?", "fehlt?") | leer.
# Begründung: Das Nutzerurteil (Explorer/foobar) ist die Referenz; Regeln aus
#   Phase 4 werden dagegen gemessen. Das Ziel gilt nur für den Stand der
#   Daten (Stick unverändert bis Projektabschluss).
# Erwartung (vorab aus der Datei gezählt, 62 Zeilen):
#   datei 20 | ordner 5 | text 17 | leer 20
#   A: datei 15, text 11, leer 10 | B: datei 1, text 3
#   C: datei 2, ordner 5, text 1, leer 10 | D: text 2 | M: datei 2
# Datenschutz: Konsole nur Zählwerte.
# =============================================================================

_AUDIO = (".mp3", ".wav", ".aif")
_zeilen = (GOLD / "05_07_ungeloeste_labels.txt").read_text(encoding="utf-8-sig").splitlines()

_rows = []
for _z in _zeilen[1:]:                       # Kopfzeile überspringen
    if not _z.strip():
        continue
    _f = (_z.split("\t") + [""] * 8)[:8]     # auf 8 Felder auffüllen
    _ziel = _f[5].strip().strip('"').strip()
    if not _ziel:
        _typ = "leer"
    elif _ziel.casefold().endswith(_AUDIO):
        _typ = "datei"
    elif _ziel[1:3] == ":\\":
        _typ = "ordner"
    else:
        _typ = "text"
    _rows.append({"fall": _f[0].strip(), "quelle": _f[1].strip(), "pfad": _f[2].strip(),
                  "ziel": _ziel, "kommentar": _f[6].strip(), "typ": _typ})

df_gold_nutzer = pd.DataFrame(_rows)
print(f"{len(df_gold_nutzer)} Zeilen")
print(pd.crosstab(df_gold_nutzer["fall"], df_gold_nutzer["typ"], margins=True).to_string())
df_gold_nutzer.to_csv(GOLD / "05_10_nutzerziele.csv", index=False,
                      encoding="utf-8-sig", sep=";")     # privat

In [ ]:
# =============================================================================
# Zelle 11 – Regeln R1 (Zeichen) und R2 (Stamm-Präfix) gegen die Nutzerziele
# =============================================================================
# Zweck: Zwei deterministische Regeln auf allen 62 ungelösten Fällen anwenden und
#   an den 20 Datei-Zielen (Gold) messen: kein Kandidat | eindeutig richtig |
#   eindeutig falsch | mehrdeutig. Fälle ohne Ziel: "eindeutig, ohne Referenz".
# Kandidaten: nur Audio-Endungen (.mp3 .wav .aif), kein AppleDouble, selber Ordner.
# R1: Länge gleich, Endung gleich; Unterschied nur Rand-Leerzeichen/Gross-Klein
#     oder genau ein Zeichen, das im gesuchten Namen ' ' oder '?' ist.
# R2: gesuchter Stamm ist echter Anfang des Kandidaten-Stamms, Endung gleich.
# Erwartung: R1 4 Treffer, R2 5 Treffer, 0 falsch eindeutig (Gold n = 20).
# Datenschutz: Konsole/Tabelle nur Zählwerte; Einzelfälle privat in reports/errors.
# =============================================================================

_AUDIO_ENDUNGEN = (".mp3", ".wav", ".aif")


def _stamm_endung(name: str) -> tuple[str, str]:
    stamm, endung = posixpath.splitext(nfc(name))
    return stamm.strip().casefold(), endung.casefold()


def regel_r1(gesucht: str, kandidaten: list[str]) -> list[str]:
    g, ge = _stamm_endung(gesucht)
    out = []
    for k in kandidaten:
        s, se = _stamm_endung(k)
        if se != ge or len(s) != len(g):
            continue
        diff = [i for i, (a, b) in enumerate(zip(g, s)) if a != b]
        if not diff or (len(diff) == 1 and g[diff[0]] in " ?\ufffd"):
            out.append(k)
    return out


def regel_r2(gesucht: str, kandidaten: list[str]) -> list[str]:
    g, ge = _stamm_endung(gesucht)
    out = []
    for k in kandidaten:
        s, se = _stamm_endung(k)
        if se == ge and len(s) > len(g) and s.startswith(g):
            out.append(k)
    return out


def ziel_relativ(ziel: str) -> str:
    s = nfc(ziel).replace("\\", "/")
    return (s[3:] if s[1:3] == ":/" else s).casefold()


_zeilen_z11 = []
for _, r in df_gold_nutzer.iterrows():
    pfad = nfc(r["pfad"]).replace("\\", "/")
    ordner, gesucht = posixpath.dirname(pfad), posixpath.basename(pfad)
    kand = [k for k in _dateien_je_ordner.get(ordner, [])
            if k.casefold().endswith(_AUDIO_ENDUNGEN) and not k.startswith("._")]
    for regel_name, regel in (("R1", regel_r1), ("R2", regel_r2)):
        treffer = regel(gesucht, kand)
        if len(treffer) == 0:
            erg = "kein"
        elif len(treffer) > 1:
            erg = "mehrdeutig"
        elif r["typ"] == "datei":
            richtig = f"{ordner}/{treffer[0]}".casefold() == ziel_relativ(r["ziel"])
            erg = "eindeutig_richtig" if richtig else "eindeutig_falsch"
        else:
            erg = "eindeutig_ohne_referenz"
        _zeilen_z11.append({"fall": r["fall"], "quelle": r["quelle"], "pfad": pfad,
                            "typ_ziel": r["typ"], "regel": regel_name,
                            "ergebnis": erg, "n_kandidaten_ordner": len(kand),
                            "treffer": "; ".join(treffer)})

df_regeln = pd.DataFrame(_zeilen_z11)
df_z11 = (df_regeln.groupby(["regel", "typ_ziel", "ergebnis"]).size()
          .rename("faelle [Eintraege]").reset_index())
print(df_z11.to_string(index=False))
_ = tabelle_speichern(df_z11, "05_11_regeln_r1_r2")
df_regeln.to_csv(ERR / "05_11_regeln_je_fall.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 12 – R2: der eine eindeutige Vorschlag ohne Referenz
# =============================================================================
# Zweck: Den Fall anzeigen, bei dem R2 einen eindeutigen Kandidaten liefert,
#   zu dem kein Nutzerziel vorliegt. Das Urteil des Nutzers (Explorer/foobar)
#   macht ihn zum Gold-Fall.
# Datenschutz: nur lokale Anzeige.
# =============================================================================

_o = df_regeln[(df_regeln["regel"] == "R2")
               & (df_regeln["ergebnis"] == "eindeutig_ohne_referenz")]
_o = _o.merge(df_gold_nutzer[["quelle", "pfad", "ziel", "kommentar"]],
              on=["quelle", "pfad"], how="left")
print(len(_o), "Fall/Fälle")
for _, r in _o.iterrows():
    print(f"Fall {r['fall']} ({r['quelle']})")
    print("  gesucht   :", r["pfad"])
    print("  Vorschlag :", r["treffer"])
    print("  dein Ziel :", repr(r["ziel"]), "| Kommentar:", repr(r["kommentar"]))

In [ ]:
# =============================================================================
# Zelle 13 – Nutzerurteil zum R2-Vorschlag ohne Referenz (Gold-Label)
# =============================================================================
# Zweck: Das Urteil des Nutzers (Explorer/foobar, 16.11.2026) als Label sichern:
#   Vorschlag der Regel R2 = dieselbe Datei (richtig).
# Begründung: Labels sind die Referenz für Phase 4; sie gelten nur für den
#   Datenstand (Stick unverändert bis Projektabschluss).
# Datenschutz: Datei privat in data/gold.
# =============================================================================

_pfad_label = GOLD / "05_13_r2_ohne_referenz_labels.csv"
# Label existiert bereits: laden statt überschreiben (Run All bleibt lauffähig)
if _pfad_label.exists():
    df_label_r2 = pd.read_csv(_pfad_label, encoding="utf-8-sig", sep=";", dtype=str)
    print(f"{len(df_label_r2)} Label geladen: {_pfad_label.name}")
else:
    df_label_r2 = _o[["quelle", "pfad", "treffer"]].assign(urteil="richtig")
    df_label_r2.to_csv(_pfad_label, index=False, encoding="utf-8-sig", sep=";")
    print(f"{len(df_label_r2)} Label gespeichert: {_pfad_label.name}")
print("R2 gegen Gold:", 5 + int((df_label_r2['urteil'] == 'richtig').sum()), "von 6 richtig")

In [ ]:
# =============================================================================
# Zelle 14 – Negativkontrolle R1/R2: Zieldatei entfernt, Vorschläge = Fehlvorschläge
# =============================================================================
# Zweck: Fehlvorschlagsrate der Regeln messen, wenn die richtige Datei fehlt.
#   Population: alle Pfade mit Klasse sicher/Stufe pfad (Crates + Playlists,
#   eindeutig). Je Pfad wird die Zieldatei aus dem Ordner-Pool entfernt.
# Lesart: "eindeutig" = Fehlvorschlag (obere Schranke; ein bitgleiches Duplikat
#   im Ordner wäre richtig). "mehrdeutig" erzeugt nie eine sichere Aktion.
# Streuung: Bootstrap über Ordner (Pfade im selben Ordner sind abhängig).
# Datenschutz: Konsole/Tabelle nur Zählwerte; Fälle privat in reports/errors.
# =============================================================================

import numpy as np

_p_c = df_rc.loc[(df_rc["klasse"] == "sicher") & (df_rc["stufe"] == "pfad"), "eintrag"]
_p_p = df_rp.loc[(df_rp["klasse"] == "sicher") & (df_rp["stufe"] == "pfad"), "kanonisch"]
_alle = sorted({nfc(str(p)).replace("\\", "/") for p in pd.concat([_p_c, _p_p])})
print(f"{len(_alle)} eindeutige sicher/pfad-Pfade (Crates {len(_p_c)}, Playlists {_p_p.nunique()})")

_z14 = []
for _p in _alle:
    _ord, _name = posixpath.dirname(_p), posixpath.basename(_p)
    _pool = [k for k in _dateien_je_ordner.get(_ord, [])
             if k.casefold().endswith(_AUDIO_ENDUNGEN) and not k.startswith("._")
             and k != _name]
    for _rn, _fn in (("R1", regel_r1), ("R2", regel_r2)):
        _t = _fn(_name, _pool)
        _z14.append({"ordner": _ord, "pfad": _p, "regel": _rn, "n_pool": len(_pool),
                     "ergebnis": "kein" if not _t else ("eindeutig" if len(_t) == 1 else "mehrdeutig"),
                     "vorschlag": _t[0] if len(_t) == 1 else ""})
df_neg = pd.DataFrame(_z14)

_rng = np.random.default_rng(SEED)
_zeilen = []
for _rn, _g in df_neg.groupby("regel"):
    _je = _g.groupby("ordner")["ergebnis"].agg(n="size", k=lambda s: (s == "eindeutig").sum())
    _n, _k = _je["n"].to_numpy(), _je["k"].to_numpy()
    _bs = []
    for _ in range(2000):
        _i = _rng.integers(0, len(_je), len(_je))
        _bs.append(_k[_i].sum() / _n[_i].sum())
    _lo, _hi = np.percentile(_bs, [2.5, 97.5])
    _c = _g["ergebnis"].value_counts()
    _zeilen.append({"regel": _rn, "pfade [Pfade]": len(_g), "ordner [Ordner]": len(_je),
                    "kein [Pfade]": int(_c.get("kein", 0)),
                    "eindeutig [Pfade]": int(_c.get("eindeutig", 0)),
                    "mehrdeutig [Pfade]": int(_c.get("mehrdeutig", 0)),
                    "fehlvorschlag [%]": round(100 * _k.sum() / _n.sum(), 2),
                    "ki95_unten [%]": round(100 * _lo, 2), "ki95_oben [%]": round(100 * _hi, 2)})
df_z14 = pd.DataFrame(_zeilen)
print(df_z14.to_string(index=False))
_ = tabelle_speichern(df_z14, "05_14_negativkontrolle_r1_r2")
df_neg[df_neg["ergebnis"] != "kein"].to_csv(
    ERR / "05_14_negativkontrolle_faelle.csv", index=False, encoding="utf-8")      # privat

In [ ]:
# =============================================================================
# Zelle 15 – R2: Fehlvorschläge der Negativkontrolle und Veto V1 (Zahlverlängerung)
# =============================================================================
# Zweck: Die 26 eindeutigen R2-Fehlvorschläge ansehen und das Veto V1 messen:
#   gesuchter Stamm endet auf Ziffer UND Anhang des Kandidaten beginnt mit Ziffer
#   -> verwerfen (nummerierter Nachbar). Wirkung auf Negativkontrolle und Gold.
# Erwartung Gold: alle 6 R2-Treffer bleiben bestehen (V1 greift dort nicht).
# Datenschutz: Konsole lokal; Zählwerte in tables/, Fälle privat.
# =============================================================================

def anhang(gesucht: str, vorschlag: str) -> tuple[str, str]:
    g, _ = _stamm_endung(gesucht)
    v, _ = _stamm_endung(vorschlag)
    return g, v[len(g):]                      # R2 garantiert: v beginnt mit g


def veto_v1(gesucht: str, vorschlag: str) -> bool:
    """True = Vorschlag verwerfen (Zahl wird nur verlängert)."""
    g, rest = anhang(gesucht, vorschlag)
    return bool(g) and g[-1].isdigit() and bool(rest) and rest[0].isdigit()


_n26 = df_neg[(df_neg["regel"] == "R2") & (df_neg["ergebnis"] == "eindeutig")].copy()
_n26["gesucht"] = _n26["pfad"].map(posixpath.basename)
_n26["verworfen_v1"] = [veto_v1(g, v) for g, v in zip(_n26["gesucht"], _n26["vorschlag"])]

_g6 = df_regeln[(df_regeln["regel"] == "R2")
                & df_regeln["ergebnis"].isin(["eindeutig_richtig", "eindeutig_ohne_referenz"])].copy()
_g6["gesucht"] = _g6["pfad"].map(posixpath.basename)
_g6["verworfen_v1"] = [veto_v1(g, v) for g, v in zip(_g6["gesucht"], _g6["treffer"])]

print(f"Negativkontrolle R2: {len(_n26)} Fehlvorschläge, "
      f"von V1 verworfen: {_n26['verworfen_v1'].sum()}, bleiben: {(~_n26['verworfen_v1']).sum()}")
print(f"Gold R2: {len(_g6)} Treffer, von V1 verworfen: {_g6['verworfen_v1'].sum()}")
print("\nFehlvorschläge (gesucht -> Vorschlag, verworfen):")
print(_n26[["gesucht", "vorschlag", "verworfen_v1"]].to_string(index=False))

df_z15 = pd.DataFrame([
    {"population": "negativkontrolle_r2_eindeutig [Pfade]", "faelle [Pfade]": len(_n26),
     "verworfen_v1 [Pfade]": int(_n26["verworfen_v1"].sum())},
    {"population": "gold_r2_treffer [Pfade]", "faelle [Pfade]": len(_g6),
     "verworfen_v1 [Pfade]": int(_g6["verworfen_v1"].sum())}])
_ = tabelle_speichern(df_z15, "05_15_veto_v1")
_n26.to_csv(ERR / "05_15_r2_fehlvorschlaege.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 16 – Negativkontrolle R2: Dateigrösse Original vs. Vorschlag
# =============================================================================
# Zweck: Für die 26 R2-Fehlvorschläge messen, ob der Vorschlag dieselbe Grösse
#   [Bytes] hat wie das (entfernte) Original. Gleiche Grösse = Kandidat für
#   Kopie (Beweis erst mit Hash); verschiedene Grösse = anderes Stück/Version.
# Begründung: Name allein trennt Umbenennung und Version nicht (Zelle 15,
#   Nutzerprüfung 20.11.2026). Prüft, ob Grösse als Inhaltsbeleg trennt.
# Kontrolle: die 2 vom Nutzer als "verschiedene Stücke" bestätigten Fälle
#   (Prüfung 20.11.2026) müssen in "verschieden" landen.
# Datenschutz: Zählwerte in tables/; Einzelfälle privat.
# =============================================================================

BYTES = dict(zip(df_ix["pfad_relativ"].map(lambda s: nfc(str(s)).replace("\\", "/")),
                 df_ix["bytes"]))

_n26["bytes_original"] = _n26["pfad"].map(BYTES)
_n26["bytes_vorschlag"] = [BYTES.get(f"{o}/{v}") for o, v in zip(_n26["ordner"], _n26["vorschlag"])]
_n26["groesse"] = np.select(
    [_n26["bytes_original"].isna() | _n26["bytes_vorschlag"].isna(),
     _n26["bytes_original"] == _n26["bytes_vorschlag"]],
    ["unbekannt", "gleich"], default="verschieden")
_n26["diff [%]"] = (100 * (_n26["bytes_vorschlag"] - _n26["bytes_original"])
                    / _n26["bytes_original"]).round(1)

print(_n26["groesse"].value_counts().to_string())
print()
print(_n26[["gesucht", "vorschlag", "groesse", "diff [%]"]].to_string(index=False))

df_z16 = _n26["groesse"].value_counts().rename("faelle [Pfade]").reset_index()
_ = tabelle_speichern(df_z16, "05_16_r2_groessenvergleich")
_n26.to_csv(ERR / "05_16_r2_groessenvergleich_faelle.csv", index=False, encoding="utf-8")  # privat

### Entscheidung 20.11.2026 – Regeln für "fehlt" mit vorhandenem Ordner (Gruppe A)

| Regel | Gold (Nutzerziele) | Negativkontrolle (2.505 Pfade, 215 Ordner) | Entscheidung |
|---|---|---|---|
| R1 Zeichenregel (Rand-Leerzeichen, Gross/Klein, 1 verlorenes Zeichen) | 4/4 richtig | 0 Fehlvorschläge (Rule of Three: < 0,12 % je Pfad, konservativ < 1,4 % je Ordner) | **sicher** (Vorbehalt: kleines n) |
| R2 Stamm-Präfix (gesuchter Name + Anhang) | 6/6 richtig | 26 Fehlvorschläge = 1,04 % (Bootstrap über Ordner, KI 0,35–1,88 %) | **Vorschlag** (P5, P13) |
| Veto V1 (Zahl wird verlängert) | 0/6 verworfen | 1/26 abgefangen | **verworfen** (kein Nutzen) |

**Befunde**
- Ein Namensanhang kennzeichnet in der Sammlung Versionen (Sicherung, Set-Bearbeitung, Nummerierung, Kopie). Name allein trennt "umbenannt" und "zweite Fassung" nicht.
- Dateigrösse: 26/26 Fehlvorschläge verschieden, 13 davon < ±1 %. Eine Toleranz bei der Grösse ist kein Beleg; nur exakte Gleichheit bzw. Hash.
- Nutzerprüfung der Fehlvorschläge: gleiches Stück / andere Datei 2, gleiches Stück / für Set bearbeitet 2, anderes Stück 3. Ein R2-Vorschlag kann gleichwertiger Ersatz, andere Fassung oder anderes Stück sein, deshalb entscheidet der Nutzer je Track.
- Im echten Betrieb fehlt die Originaldatei; ein Inhaltsbeleg ist nur über Metadaten der Quelle möglich (Serato `database V2`: Grösse/Dauer; M3U: EXTINF-Dauer). Prüfung folgt.

In [ ]:
# =============================================================================
# Zelle 17 – Negativkontrolle R2: hätte Stufe 4.3a (Name anderswo) vorher gegriffen?
# =============================================================================
# Zweck: Für die 26 R2-Fehlvorschläge (Original versteckt) prüfen, ob eine
#   Datei mit gleichem Namen in einem anderen Ordner existiert:
#   0 = keine (R2 käme zum Zug) | 1 = eindeutig (Stufe 4.3a, Vorschlag)
#   | >1 = mehrfach. Zusätzlich: Grösse gleich Original (Hinweis auf Kopie).
# Begründung: Misst, wie oft R2 im Ernstfall überhaupt erreicht wird
#   (Nutzerhinweis 20.11.2026: gesuchte Datei liegt teils zusätzlich anderswo).
# Vergleich: Dateiname exakt (NFC); Gross/Klein separat ausgewiesen.
# Datenschutz: Zählwerte in tables/; Einzelfälle privat.
# =============================================================================

_name_zu_pfaden = defaultdict(list)
_fold_zu_pfaden = defaultdict(list)
for _p in _pfade:
    _b = posixpath.basename(_p)
    _name_zu_pfaden[_b].append(_p)
    _fold_zu_pfaden[_b.casefold()].append(_p)

_z17 = []
for _, r in _n26.iterrows():
    _orig, _name = r["pfad"], posixpath.basename(r["pfad"])
    _exakt = [p for p in _name_zu_pfaden.get(_name, []) if p != _orig]
    _fold = [p for p in _fold_zu_pfaden.get(_name.casefold(), []) if p != _orig]
    _b_orig = BYTES.get(_orig)
    _z17.append({
        "gesucht": _name,
        "anderswo_exakt [Dateien]": len(_exakt),
        "anderswo_grossklein [Dateien]": len(_fold),
        "davon_groesse_gleich [Dateien]": sum(BYTES.get(p) == _b_orig for p in _fold),
        "fundorte": "; ".join(posixpath.dirname(p) for p in _fold)})
df_z17_faelle = pd.DataFrame(_z17)

def _stufe(n: int) -> str:
    return "0: R2 käme zum Zug" if n == 0 else ("1: Stufe 4.3a eindeutig" if n == 1 else ">1: mehrfach")

df_z17_faelle["vor_r2"] = df_z17_faelle["anderswo_grossklein [Dateien]"].map(_stufe)
df_z17 = (df_z17_faelle.groupby("vor_r2")
          .agg(**{"faelle [Pfade]": ("gesucht", "size"),
                  "groesse_gleich [Pfade]": ("davon_groesse_gleich [Dateien]", lambda s: (s > 0).sum())})
          .reset_index())
print(df_z17.to_string(index=False))
print()
print(df_z17_faelle.loc[df_z17_faelle["anderswo_grossklein [Dateien]"] > 0,
      ["gesucht", "anderswo_exakt [Dateien]", "anderswo_grossklein [Dateien]",
       "davon_groesse_gleich [Dateien]", "fundorte"]].to_string(index=False))
_ = tabelle_speichern(df_z17, "05_17_kaskade_vor_r2")
df_z17_faelle.to_csv(ERR / "05_17_kaskade_vor_r2_faelle.csv", index=False, encoding="utf-8")  # privat

In [ ]:
# =============================================================================
# Zelle 18 – Inventar der Artefakte aus NB02/NB03 (Serato-Metadaten, Tags)
# =============================================================================
# Zweck: Dateien finden, die Serato database V2 (Grösse/Dauer je Track) und
#   Tags (Dauer je Datei) enthalten, als Grundlage für Inhaltsbelege zu R2.
# Begründung: Dateinamen werden gemessen, nicht angenommen.
# Datenschutz: nur Dateinamen und Grösse.
# =============================================================================

_inv = [{"ordner": n, "datei": p.name, "groesse [KB]": round(p.stat().st_size / 1024, 1)}
        for n, o in {"interim": INTERIM, "raw": RAW}.items()
        for p in sorted(o.rglob("*"))
        if p.is_file() and (p.name.startswith(("02_", "03_")) or "database" in p.name.lower()
                            or "serato" in str(p.parent).lower())]
df_inv18 = pd.DataFrame(_inv)
print(df_inv18.to_string(index=False))

In [ ]:
# =============================================================================
# Zelle 19 – Struktur Serato database V2 (NB02 Z.08) und Audio-Tags (NB02 Z.12)
# =============================================================================
# Zweck: Spalten für Inhaltsbelege feststellen (Grösse, Dauer je Track/Datei).
# Begründung: Spaltennamen werden gemessen, nicht angenommen; keine Zeilen (P11).
# =============================================================================

for _name in ("02_08_database_v2.csv", "02_12_audio_tags.csv"):
    _df = pd.read_csv(INTERIM / _name, low_memory=False)
    print(f"--- {_name}: {len(_df)} Zeilen")
    print(_df.dtypes.to_string())
    print("Anteil gefüllt [%]:", (_df.notna().mean() * 100).round(1).to_dict())
    print()

In [ ]:
# =============================================================================
# Zelle 20 – Format der Serato-Felder tsiz (Grösse) und tlen (Länge)
# =============================================================================
# Zweck: Textformat feststellen, bevor die Felder in Zahlen umgerechnet und mit
#   Index (bytes) und Tags (duration) verglichen werden.
# Begründung: Format wird gemessen, nicht angenommen. Nur Zahlenwerte (P11).
# =============================================================================

df_db = pd.read_csv(INTERIM / "02_08_database_v2.csv", low_memory=False)
for _sp in ("tsiz", "tlen"):
    _w = df_db[_sp].dropna()
    print(f"--- {_sp}: {len(_w)} Werte, Beispiele:")
    print(_w.sample(8, random_state=SEED).to_string(index=False))
    print("Muster (Ziffern -> 9):",
          _w.str.replace(r"\d", "9", regex=True).value_counts().head(5).to_dict())
    print()

In [ ]:
# =============================================================================
# Zelle 21 – Kalibrierung Serato tsiz/tlen gegen Index (bytes) und Tags (duration)
# =============================================================================
# Zweck: An vorhandenen Dateien messen, (a) ob "MB" 10^6 oder 2^20 Byte meint,
#   (b) wie gross die Abweichung tlen <-> duration ist (Grundlage für eine
#   Toleranz als Inhaltsbeleg in R2).
# Zuordnung: Spalte 'schluessel' (NB02 Z.08) gegen kanonischen Indexpfad.
# Datenschutz: nur Zählwerte und Verteilungen.
# =============================================================================

def tsiz_mb(s) -> float:
    return float(s[:-2]) if isinstance(s, str) and s.endswith("MB") else np.nan


def tlen_s(s) -> float:
    if not isinstance(s, str) or ":" not in s:
        return np.nan
    m, sek = s.split(":")
    return int(m) * 60 + float(sek)


df_tags = pd.read_csv(INTERIM / "02_12_audio_tags.csv", low_memory=False)
DAUER = dict(zip(df_tags["pfad_relativ"].map(lambda s: nfc(str(s)).replace("\\", "/")),
                 df_tags["duration"]))

_k = df_db.copy()
_k["key"] = _k["schluessel"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_k["mb"], _k["sek_db"] = _k["tsiz"].map(tsiz_mb), _k["tlen"].map(tlen_s)
_k["bytes"], _k["sek_tag"] = _k["key"].map(BYTES), _k["key"].map(DAUER)
print(f"Serato-Tracks: {len(_k)} | im Index gefunden: {_k['bytes'].notna().sum()}")

_g = _k.dropna(subset=["mb", "bytes"])
for _n, _f in (("10^6", 1e6), ("2^20", 2**20)):
    _d = (_g["bytes"] / _f - _g["mb"]).abs()
    print(f"tsiz als {_n}: n={len(_d)}, |diff| <= 0,05 MB: {(_d <= 0.05 + 1e-9).mean():.1%}, "
          f"Median {_d.median():.3f} MB")

_t = _k.dropna(subset=["sek_db", "sek_tag"])
_dt = (_t["sek_db"] - _t["sek_tag"]).abs()
print(f"\ntlen vs duration: n={len(_dt)}")
print(_dt.describe(percentiles=[.5, .9, .95, .99]).round(3).to_string())
for _s in (0.05, 0.1, 0.5, 1.0):
    print(f"|diff| <= {_s} s: {(_dt <= _s).mean():.1%}")

df_z21 = pd.DataFrame({"toleranz [s]": [0.05, 0.1, 0.5, 1.0],
                       "anteil [%]": [round(100 * (_dt <= s).mean(), 1) for s in (0.05, 0.1, 0.5, 1.0)],
                       "n [Dateien]": len(_dt)})
_ = tabelle_speichern(df_z21, "05_21_kalibrierung_tlen")

In [ ]:
# =============================================================================
# Zelle 22 – Inhaltsbeleg für R2: Serato-Metadaten des gesuchten Titels vs. Kandidat
# =============================================================================
# Zweck: Für Gold-R2-Treffer (6, richtig) und Negativkontrolle (26, falsch)
#   prüfen, ob tlen/tsiz des gesuchten Pfads (database V2) zu Dauer/Grösse des
#   vorgeschlagenen Kandidaten passen.
# Toleranzen (Zelle 21): Dauer <= 0,1 s (92,4 % gleiche Datei), Grösse <= 0,05 MiB.
# Ergebnis je Fall: bestaetigt | widerspruch | kein_beleg (keine Serato-Daten).
# Datenschutz: Zählwerte in tables/; Einzelfälle privat.
# =============================================================================

META = {r.key: (r.sek_db, r.mb) for r in _k.itertuples()}


def beleg(gesucht_pfad: str, kandidat_pfad: str) -> dict:
    sek_db, mb = META.get(gesucht_pfad, (np.nan, np.nan))
    sek_k, b_k = DAUER.get(kandidat_pfad, np.nan), BYTES.get(kandidat_pfad, np.nan)
    d_ok = abs(sek_db - sek_k) <= 0.1 if pd.notna(sek_db) and pd.notna(sek_k) else None
    g_ok = abs(b_k / 2**20 - mb) <= 0.05 + 1e-9 if pd.notna(mb) and pd.notna(b_k) else None
    werte = [v for v in (d_ok, g_ok) if v is not None]
    erg = "kein_beleg" if not werte else ("bestaetigt" if all(werte) else "widerspruch")
    return {"dauer_ok": d_ok, "groesse_ok": g_ok, "ergebnis": erg,
            "diff_dauer [s]": round(sek_k - sek_db, 2) if d_ok is not None else np.nan}


_rows22 = []
for _, r in _g6.iterrows():
    kand = f"{posixpath.dirname(r['pfad'])}/{r['treffer']}"
    _rows22.append({"population": "gold_richtig", "gesucht": r["gesucht"],
                    "vorschlag": r["treffer"], **beleg(r["pfad"], kand)})
for _, r in _n26.iterrows():
    _rows22.append({"population": "negativ_falsch", "gesucht": r["gesucht"],
                    "vorschlag": r["vorschlag"],
                    **beleg(r["pfad"], f"{r['ordner']}/{r['vorschlag']}")})
df_b22 = pd.DataFrame(_rows22)

df_z22 = pd.crosstab(df_b22["population"], df_b22["ergebnis"], margins=True)
print(df_z22.to_string())
print()
print(df_b22[["population", "gesucht", "vorschlag", "dauer_ok", "groesse_ok",
              "diff_dauer [s]", "ergebnis"]].to_string(index=False))
_ = tabelle_speichern(df_z22.reset_index(), "05_22_r2_inhaltsbeleg_serato")
df_b22.to_csv(ERR / "05_22_r2_inhaltsbeleg_faelle.csv", index=False, encoding="utf-8")  # privat

### Entscheidung 20.11.2026 – Inhaltsbeleg für R2 (Serato database V2)

**Kalibrierung (Zelle 21, vorhandene Dateien):** `tsiz` in MiB (2^20), Auflösung 0,1 MiB; 76,2 % innerhalb der Rundung. `tlen` vs. Tag-Dauer: Median 0,003 s, 92,4 % <= 0,1 s, 96,8 % <= 1 s.

**Test (Zelle 22), Beleg = Dauer <= 0,1 s UND Grösse <= 0,05 MiB:**

| Population | bestätigt | Widerspruch |
|---|---|---|
| Gold richtig (6) | 3 | 3 |
| Negativkontrolle falsch (26) | 0 | 26 |

- Dauer allein: Gold 5/6, aber 1/26 falsch bestätigt (gleiche Dauer, andere Grösse, Hypothese: andere Bitrate).
- Toleranz "Kandidat bis 4 s kürzer" (Nutzer schneidet Titel nachträglich): Gold 6/6, aber 2/26 falsch bestätigt.
- Ursache: Titel werden nach dem Serato-Eintrag bearbeitet (Schnitt, Tags). Gleiches Stück (geschnitten) und andere Version (ähnliche Länge) sind über Name, Grösse und Dauer nicht sicher trennbar.

**Entscheidung:** R2 bleibt Vorschlag. Der Serato-Abgleich (Abweichung Dauer/Grösse) erscheint als Hinweis je Vorschlag im Bericht. Keine grosse Negativkontrolle / kein ML-Paar-Klassifikator für diesen Fall: Die Merkmale trennen nicht, eine grössere Stichprobe ändert das nicht.

**Leitfrage:** Für umbenannte und bearbeitete Dateien liefert kein verfügbarer Inhaltsbeleg eine sichere Zuordnung. Entscheider bleibt der Nutzer.

In [ ]:
# =============================================================================
# Zelle 23 – Gruppen B/C/D/M: Ordner durch 0-Byte-Datei ersetzt?
# =============================================================================
# Zweck: Prüfen, ob ein Ordner im Pfad einer 0-Byte-Datei (ohne Audio-Endung,
#   an Stelle des Ordners) entspricht -> Klasse album_beschaedigt statt fehlt.
#   Fehlendes Präfix 'Musik/' wird ergänzt (Präfix-Fehler fremder Pfade).
# Stufen: exakt (Pfad gleich) | lose (Name casefold, nur Buchstaben/Ziffern).
# Erwartung: exakt 4, lose zusätzlich 3, zusammen 7 (alle Gruppe C); B/D/M 0.
# Datenschutz: Zählwerte in tables/; Einzelfälle nur lokal und privat.
# =============================================================================

_null = df_ix[(df_ix["bytes"] == 0)
              & ~df_ix["pfad_relativ"].str.lower().str.endswith(_AUDIO_ENDUNGEN)]
NULL_EXAKT = {nfc(str(p)).replace("\\", "/") for p in _null["pfad_relativ"]}


def _lose(s: str) -> str:
    return re.sub(r"[^0-9a-z]", "", s.casefold())


NULL_LOSE = {_lose(posixpath.basename(p)): p for p in NULL_EXAKT}
print(f"0-Byte-Dateien ohne Audio-Endung im Index: {len(NULL_EXAKT)}")


def null_ordner(pfad: str) -> tuple[str, str]:
    p = nfc(str(pfad)).replace("\\", "/")
    if not p.startswith("Musik/"):
        p = "Musik/" + p
    teile = p.split("/")[:-1]
    for i in range(2, len(teile) + 1):
        if "/".join(teile[:i]) in NULL_EXAKT:
            return "exakt", "/".join(teile[:i])
    for t in teile[1:]:
        if _lose(t) in NULL_LOSE:
            return "lose", NULL_LOSE[_lose(t)]
    return "nein", ""


_bcdm = df_gold_nutzer[df_gold_nutzer["fall"].isin(["B", "C", "D", "M"])].copy()
_bcdm[["null_treffer", "null_datei"]] = [null_ordner(p) for p in _bcdm["pfad"]]

print("Fall: B Ordner anderswo 1x | C kein gleichnamiger Ordner, ohne 'Musik/' "
      "| D kein Ordner, mit 'Musik/' | M mehrere Kandidaten")
print("Treffer: exakt = Ordner im Pfad ist 0-Byte-Datei gleichen Namens | "
      "lose = dito, Name ohne Gross/Klein/Satzzeichen | nein = keine 0-Byte-Datei\n")
df_z23 = pd.crosstab(_bcdm["fall"], _bcdm["null_treffer"], margins=True)
print(df_z23.to_string())
print()
print(_bcdm.loc[_bcdm["null_treffer"] != "nein", ["fall", "pfad", "null_treffer", "null_datei"]]
      .to_string(index=False))
_ = tabelle_speichern(df_z23.reset_index(), "05_23_bcdm_nullbyte_ordner")
_bcdm.to_csv(ERR / "05_23_bcdm_nullbyte_faelle.csv", index=False, encoding="utf-8")   # privat

### Entscheidung 21.11.2026 – Präfix-Ergänzung und 0-Byte-Ordner

**Befund (Zelle 23):** 11 von 18 Pfaden der Gruppe C (ohne Präfix `Musik/`, kein gleichnamiger Ordner) führen nach Präfix-Ergänzung auf eine 0-Byte-Datei an Stelle des Ordners: exakt 8 (2 Ordner), lose 3 (1 Ordner, Name nur ohne Gross/Klein/Satzzeichen gleich). Nutzer bestätigt beide Fälle (Explorer). Gruppen B, D, M: 0 Treffer.

**Regel:**
- Treffer exakt -> Klasse `album_beschaedigt` (statt `fehlt`)
- Treffer lose -> Klasse `album_beschaedigt` mit Hinweis "Ordnername nur ähnlich"

**Begründung:** Die Regel ordnet keine Datei zu, sie präzisiert nur den Grund für einen offenen Eintrag (P5 nicht berührt). Nutzen: Bericht und Bonus "Suche auf Sicherungen" nennen den verlorenen Ordner.

**Umsetzung:** `resolver` (src) mit Test, Phase 4.

**Stand ungelöste Fälle (62):** album_beschaedigt 11 | offen: A 36, B 4, C 7, D 2, M 2.

In [ ]:
# =============================================================================
# Zelle 24 – Gruppe M: Kandidaten gleichen Namens im Vergleich
# =============================================================================
# Zweck: Für die 2 Mehrfach-Fälle alle Kandidaten mit Ordner, Grösse, Dauer,
#   Nutzerziel und Ordner-Ähnlichkeit (Jaccard der Wörter/Zahlen zwischen
#   Playlist-Ordner und Kandidaten-Ordner) zeigen.
# Begründung: Prüft, ob (a) Kandidaten Kopien sind (dann jeder richtig) oder
#   (b) der Ordnername das Ziel eindeutig bestimmt (Regel-Kandidat).
# Datenschutz: nur lokale Anzeige; Datei privat.
# =============================================================================

def _woerter(s: str) -> set:
    return set(re.findall(r"[0-9a-z]+", nfc(s).casefold()))


def jaccard(a: str, b: str) -> float:
    wa, wb = _woerter(a), _woerter(b)
    return round(len(wa & wb) / len(wa | wb), 3) if wa | wb else 0.0


print("Legende: ziel = Kandidat ist das Nutzerziel | jaccard = Anteil gemeinsamer "
      "Wörter/Zahlen der Ordnernamen (1 = gleich, 0 = nichts gemeinsam)\n")
_rows24 = []
for _, r in df_gold_nutzer[df_gold_nutzer["fall"] == "M"].iterrows():
    _p = nfc(r["pfad"]).replace("\\", "/")
    _ord_pl = posixpath.basename(posixpath.dirname(_p))
    for _k in _name_zu_pfaden.get(posixpath.basename(_p), []):
        _rows24.append({"fall_nr": r["pfad"][-25:], "kandidat_ordner": posixpath.dirname(_k),
                        "bytes": BYTES.get(_k), "dauer [s]": round(DAUER.get(_k, np.nan), 2),
                        "ziel": _k.casefold() == ziel_relativ(r["ziel"]),
                        "jaccard": jaccard(_ord_pl, posixpath.basename(posixpath.dirname(_k)))})
df_m24 = pd.DataFrame(_rows24)
print(df_m24.to_string(index=False))
df_m24.to_csv(ERR / "05_24_gruppe_m_kandidaten.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 25 – (ersetzt) SHA-256 der Kandidaten vom Stick
# =============================================================================
# Ersetzt durch Zelle 26: Datei-Hash aus dem Integritäts-Manifest (NB04 Z.08),
#   ohne erneutes Lesen des Sticks (Entscheidung 21.11.2026, Artefakte statt
#   physischer Messung). Erster Aufruf scheiterte an der Schnittstelle
#   anchor.finde_stick(signatur).
# Verbleibend: Import für Zelle 27.
# =============================================================================

import hashlib

In [ ]:
# =============================================================================
# Zelle 26 – Gruppe M: Hash aus dem Integritäts-Manifest (NB04 Z.08), ohne Stick
# =============================================================================
# Zweck: Bitgleichheit der je 3 Kandidaten über das gespeicherte Manifest
#   prüfen, statt den Stick erneut zu lesen.
# Begründung: Teure/physische Messungen einmal erzeugen, danach nur Artefakte
#   lesen (P7, Reproduzierbarkeit). Spalten werden gemessen, nicht angenommen.
# Legende: hash_vorhanden = Manifest enthält Hash für den Kandidaten |
#   bitgleich = alle Kandidaten eines Falls haben denselben Hash.
# Datenschutz: Hash gekürzt, nur lokale Anzeige.
# =============================================================================

df_man = pd.read_csv(INTERIM / "04_08_manifest_stick.csv", low_memory=False)
print(f"Manifest: {len(df_man)} Zeilen")
print(df_man.dtypes.to_string())

_sp_pfad = next(c for c in df_man.columns if "pfad" in c.lower())
_sp_hash = next((c for c in df_man.columns if "sha" in c.lower() or "hash" in c.lower()), None)
print(f"\nPfadspalte: {_sp_pfad} | Hashspalte: {_sp_hash}")

if _sp_hash:
    HASH = dict(zip(df_man[_sp_pfad].map(lambda s: nfc(str(s)).replace("\\", "/")),
                    df_man[_sp_hash]))
    _pl = dict(zip(df_gold_nutzer["pfad"].str[-25:], df_gold_nutzer["pfad"]))
    df_m24["hash"] = [HASH.get(f"{o}/{posixpath.basename(_pl[f])}")
                      for o, f in zip(df_m24["kandidat_ordner"], df_m24["fall_nr"])]
    df_m24["hash_vorhanden"] = df_m24["hash"].map(lambda h: isinstance(h, str) and len(h) > 0)
    df_m24["hash_kurz"] = df_m24["hash"].astype(str).str[:12]
    print(df_m24[["fall_nr", "kandidat_ordner", "ziel", "jaccard",
                  "hash_vorhanden", "hash_kurz"]].to_string(index=False))
    print("\nbitgleich je Fall:",
          df_m24.groupby("fall_nr")["hash"].nunique(dropna=True).eq(1).to_dict())

In [ ]:
# =============================================================================
# Zelle 27 – Gruppe M: Audio-Hash ohne Tags (Hypothese: nur Tags verschieden)
# =============================================================================
# Zweck: SHA-256 nur über die Audiodaten (ohne ID3v2 am Anfang, APEv2/ID3v1 am
#   Ende) der 6 Kandidaten. Gleicher Audio-Hash bei verschiedenem Datei-Hash
#   belegt: gleiche Musik, nur Tags geändert.
# Stick: Laufwerk wird über die Existenz aller 6 Dateien bestimmt (kein Raten
#   der anchor-Schnittstelle; Umstellung auf anchor beim Refactoring).
# Rein lesend (P1/P7-Ausnahme Hash).
# Legende: datei_hash = ganze Datei (Manifest) | audio_hash = ohne Tags |
#   tag_bytes = übersprungene Bytes (Kopf + Ende).
# Datenschutz: Hash gekürzt, nur lokale Anzeige.
# =============================================================================

import string


def audio_bereich(daten: bytes) -> tuple[int, int]:
    """Start/Ende der Audiodaten ohne ID3v2 (Kopf), APEv2 und ID3v1 (Ende)."""
    start, ende = 0, len(daten)
    if daten[:3] == b"ID3" and len(daten) >= 10:
        groesse = ((daten[6] & 0x7F) << 21 | (daten[7] & 0x7F) << 14
                   | (daten[8] & 0x7F) << 7 | (daten[9] & 0x7F))
        start = 10 + groesse + (10 if daten[5] & 0x10 else 0)   # Footer-Flag
    if ende - start >= 128 and daten[ende - 128:ende - 125] == b"TAG":
        ende -= 128                                             # ID3v1
    if ende - start >= 32 and daten[ende - 32:ende - 24] == b"APETAGEX":
        ape = int.from_bytes(daten[ende - 20:ende - 16], "little")
        kopf = 32 if int.from_bytes(daten[ende - 12:ende - 8], "little") & 0x80000000 else 0
        ende -= ape + kopf                                      # APEv2
    return start, ende


_pl = dict(zip(df_gold_nutzer["pfad"].str[-25:], df_gold_nutzer["pfad"]))
_rel = [f"{o}/{posixpath.basename(_pl[f])}"
        for o, f in zip(df_m24["kandidat_ordner"], df_m24["fall_nr"])]
_lw = [l for l in string.ascii_uppercase
       if all((Path(f"{l}:/") / r).is_file() for r in _rel)]
assert len(_lw) == 1, f"Laufwerk nicht eindeutig: {_lw}"
print("Stick-Laufwerk:", _lw[0])

_ah, _tb = [], []
for r in _rel:
    d = (Path(f"{_lw[0]}:/") / r).read_bytes()
    s, e = audio_bereich(d)
    _ah.append(hashlib.sha256(d[s:e]).hexdigest())
    _tb.append(len(d) - (e - s))
df_m24["audio_hash_kurz"] = [h[:12] for h in _ah]
df_m24["tag_bytes"] = _tb
df_m24["audio_hash"] = _ah

print(df_m24[["fall_nr", "kandidat_ordner", "ziel", "hash_kurz",
              "audio_hash_kurz", "tag_bytes"]].to_string(index=False))
print("\nje Fall: Datei-Hashes verschieden / Audio-Hashes verschieden")
print(df_m24.groupby("fall_nr").agg(datei_hashes=("hash", "nunique"),
                                    audio_hashes=("audio_hash", "nunique")).to_string())

### Entscheidung 21.11.2026 – Mehrfach-Kandidaten: Audio-Hash ohne Tags

**Befund (Zellen 24–27, Gruppe M, 2 Fälle à 3 Kandidaten):** Grösse und Dauer je Fall gleich, Datei-Hash verschieden (3 bzw. 2 Werte), Audio-Hash ohne ID3v2/APEv2/ID3v1 je Fall identisch (1 Wert). Hypothese bestätigt: gleiche Musik, nur Tags geändert (gleich langer Tag-Bereich). Nutzerurteil: Kopien zu Sicherungszwecken, Original im Set-Ordner. Ordner-Jaccard: Ziel 0,667 / 0,444, Kopien 0,000.

**Regel `mehrfach_audiogleich`:**
- Alle Kandidaten mit gleichem Audio-Hash -> Identität **sicher**; Ort = Kandidat mit höchstem Ordner-Jaccard, wenn eindeutig vorne, sonst Vorschlag. Ort im Bericht, per P13 änderbar.
- Audio-Hashes verschieden (z. B. eine Kopie nachträglich gekürzt) -> **Vorschlag** mit Versionsvergleich (Dauer, Änderungsdatum, Ordner); Auswahl durch den Nutzer.
- Die Regel wählt nie zwischen Versionen, nur zwischen identischen Kopien.

**Umsetzung:** Audio-Hash im Index (src, mp3), Tests mit synthetischen Dateien (ohne Tag, ID3v1, ID3v2 mit/ohne Footer, APEv2). Danach Messung über die Sammlung: zusätzliche Dubletten und Häufigkeit "Versionen verschieden".

**Leitfrage:** Inhaltsgleichheit trotz bearbeiteter Tags ist deterministisch beweisbar; ML ist dafür nicht nötig.

In [ ]:
# =============================================================================
# Zelle 28 – Regel R3 "Tracknummer im Ordner" auf die offenen Fälle A/B/C/D
# =============================================================================
# Zweck: Fälle mit führender Tracknummer im Dateinamen über die Nummer im
#   (vorhandenen oder per Ordner-Jaccard eindeutig bestimmten) Ordner zuordnen.
# Zielordner: vorhanden -> dieser | sonst höchster Jaccard der Ordnernamen,
#   nur wenn >= 0,5 und eindeutig vorne (kein Gleichstand).
# Erwartung Gold: A 2/2 richtig; C keine eindeutige Ordnerzuordnung
#   (Gleichstand 0,667 zwischen zwei Ordnern).
# Legende: ordner_status = vorhanden | jaccard (eindeutig) | gleichstand |
#   zu_unaehnlich | ohne_nummer. ergebnis = kein | eindeutig_richtig |
#   eindeutig_falsch | eindeutig_ohne_referenz | mehrdeutig.
# Datenschutz: Zählwerte in tables/; Einzelfälle privat.
# =============================================================================

_NUM = re.compile(r"^(\d{1,3})(?=[\s_.\-])")
_ORDNER_LISTE = sorted(ORDNER_IDX)


def tracknummer(name: str):
    m = _NUM.match(nfc(name))
    return int(m.group(1)) if m else None


def zielordner(ordner_pl: str) -> tuple[str, str]:
    p = ordner_pl if ordner_pl.startswith("Musik/") else "Musik/" + ordner_pl
    if p in ORDNER_IDX:
        return "vorhanden", p
    b = posixpath.basename(ordner_pl)
    werte = sorted(((jaccard(b, posixpath.basename(o)), o) for o in _ORDNER_LISTE), reverse=True)
    if werte[0][0] < 0.5:
        return "zu_unaehnlich", ""
    if len(werte) > 1 and werte[1][0] == werte[0][0]:
        return "gleichstand", ""
    return "jaccard", werte[0][1]


_offen = df_gold_nutzer[df_gold_nutzer["fall"].isin(["A", "B", "C", "D"])].copy()
_offen = _offen[~_offen["pfad"].isin(_bcdm.loc[_bcdm["null_treffer"] != "nein", "pfad"])]

_rows28 = []
for _, r in _offen.iterrows():
    p = nfc(r["pfad"]).replace("\\", "/")
    nr = tracknummer(posixpath.basename(p))
    if nr is None:
        st, zo, erg, tr = "ohne_nummer", "", "kein", []
    else:
        st, zo = zielordner(posixpath.dirname(p))
        tr = [k for k in _dateien_je_ordner.get(zo, [])
              if k.casefold().endswith(_AUDIO_ENDUNGEN) and not k.startswith("._")
              and tracknummer(k) == nr] if zo else []
        if not tr:
            erg = "kein"
        elif len(tr) > 1:
            erg = "mehrdeutig"
        elif r["typ"] == "datei":
            erg = ("eindeutig_richtig" if f"{zo}/{tr[0]}".casefold() == ziel_relativ(r["ziel"])
                   else "eindeutig_falsch")
        else:
            erg = "eindeutig_ohne_referenz"
    _rows28.append({"fall": r["fall"], "pfad": p, "typ_ziel": r["typ"], "ordner_status": st,
                    "zielordner": zo, "ergebnis": erg, "treffer": "; ".join(tr)})
df_r3 = pd.DataFrame(_rows28)

print(f"{len(df_r3)} offene Fälle (ohne 0-Byte-Ordner)\n")
print(pd.crosstab([df_r3["fall"], df_r3["ordner_status"]], df_r3["ergebnis"], margins=True).to_string())
_ = tabelle_speichern(df_r3.groupby(["fall", "ordner_status", "ergebnis"]).size()
                      .rename("faelle [Eintraege]").reset_index(), "05_28_regel_r3")
df_r3.to_csv(ERR / "05_28_regel_r3_faelle.csv", index=False, encoding="utf-8")       # privat

In [ ]:
# =============================================================================
# Zelle 29 – R3: Vorschläge ohne Referenz und mehrdeutige Fälle zur Prüfung
# =============================================================================
# Zweck: Nutzerprüfung (Explorer/foobar) der 11 eindeutigen R3-Vorschläge ohne
#   Nutzerziel und der 2 mehrdeutigen Fälle -> Gold-Labels.
# Legende: nr = laufende Nummer für die Antwort | gesucht = Dateiname aus der
#   Playlist | vorschlag = Treffer(s) der Regel R3 im Zielordner.
# Datenschutz: nur lokale Anzeige.
# =============================================================================

df_r3_pruef = (df_r3[df_r3["ergebnis"].isin(["eindeutig_ohne_referenz", "mehrdeutig"])]
               .assign(gesucht=lambda d: d["pfad"].map(posixpath.basename))
               .sort_values(["ergebnis", "pfad"]).reset_index(drop=True))
df_r3_pruef.insert(0, "nr", range(1, len(df_r3_pruef) + 1))
print(df_r3_pruef[["nr", "ergebnis", "zielordner", "gesucht", "treffer"]].to_string(index=False))

In [ ]:
# =============================================================================
# Zelle 30 – Gold-Labels R3 (Nutzerprüfung 21.11.2026: Nr. 1–11 richtig)
# =============================================================================
# Zweck: Urteile zu den 11 eindeutigen R3-Vorschlägen ohne Referenz sichern;
#   mehrdeutige Fälle (Sammelordner, Nummer 01 = Set-Teil) bleiben "fehlt".
# Datenschutz: Datei privat in data/gold; existiert sie, wird sie geladen.
# =============================================================================

_pfad_r3 = GOLD / "05_30_r3_labels.csv"
if _pfad_r3.exists():
    df_label_r3 = pd.read_csv(_pfad_r3, encoding="utf-8-sig", sep=";", dtype=str)
    print(f"{len(df_label_r3)} Labels geladen: {_pfad_r3.name}")
else:
    df_label_r3 = (df_r3_pruef[df_r3_pruef["ergebnis"] == "eindeutig_ohne_referenz"]
                   [["nr", "pfad", "zielordner", "treffer"]].assign(urteil="richtig"))
    df_label_r3.to_csv(_pfad_r3, index=False, encoding="utf-8-sig", sep=";")
    print(f"{len(df_label_r3)} Labels gespeichert: {_pfad_r3.name}")
print("R3 gegen Gold: 2 (Nutzerziele) +", (df_label_r3["urteil"] == "richtig").sum(),
      "von", 2 + len(df_label_r3), "richtig")

In [ ]:
# =============================================================================
# Zelle 31 – Negativkontrolle R3: Original entfernt, gleiche Tracknummer im Ordner?
# =============================================================================
# Zweck: Fehlvorschlagsrate von R3 messen. Population: sicher/pfad-Pfade
#   (Zelle 14) mit führender Tracknummer. Original wird aus dem Ordner-Pool
#   entfernt; eindeutiger Treffer = Fehlvorschlag, ausser bitgleiche Kopie
#   (gleicher SHA-256 laut Manifest).
# Legende: kein = keine Datei mit gleicher Nummer | eindeutig_fremd =
#   Fehlvorschlag | eindeutig_kopie = bitgleiche Kopie (inhaltlich richtig)
#   | mehrdeutig = mehrere Treffer (keine Aktion).
# Streuung: Bootstrap über Ordner (2.000 Wiederholungen, SEED).
# Datenschutz: Zählwerte in tables/; Fälle privat.
# =============================================================================

_rows31 = []
for _p in _alle:
    _ord, _name = posixpath.dirname(_p), posixpath.basename(_p)
    _nr = tracknummer(_name)
    if _nr is None:
        continue
    _t = [k for k in _dateien_je_ordner.get(_ord, [])
          if k != _name and k.casefold().endswith(_AUDIO_ENDUNGEN)
          and not k.startswith("._") and tracknummer(k) == _nr]
    if not _t:
        _erg = "kein"
    elif len(_t) > 1:
        _erg = "mehrdeutig"
    else:
        _gleich = HASH.get(_p) is not None and HASH.get(_p) == HASH.get(f"{_ord}/{_t[0]}")
        _erg = "eindeutig_kopie" if _gleich else "eindeutig_fremd"
    _rows31.append({"ordner": _ord, "pfad": _p, "ergebnis": _erg, "vorschlag": "; ".join(_t[:3])})
df_neg3 = pd.DataFrame(_rows31)

_je = df_neg3.groupby("ordner")["ergebnis"].agg(n="size", k=lambda s: (s == "eindeutig_fremd").sum())
_n, _k = _je["n"].to_numpy(), _je["k"].to_numpy()
_rng = np.random.default_rng(SEED)
_bs = [(_k[i].sum() / _n[i].sum())
       for i in (_rng.integers(0, len(_je), len(_je)) for _ in range(2000))]
_lo, _hi = np.percentile(_bs, [2.5, 97.5])

print(f"Population: {len(df_neg3)} Pfade mit Tracknummer in {len(_je)} Ordnern "
      f"(von {len(_alle)} sicher/pfad)")
print(df_neg3["ergebnis"].value_counts().to_string())
print(f"\nFehlvorschlag: {100 * _k.sum() / _n.sum():.2f} % "
      f"(Bootstrap-KI über Ordner {100 * _lo:.2f}–{100 * _hi:.2f} %)")

df_z31 = (df_neg3["ergebnis"].value_counts().rename("faelle [Pfade]").reset_index()
          .assign(**{"fehlvorschlag [%]": round(100 * _k.sum() / _n.sum(), 2),
                     "ki95_unten [%]": round(100 * _lo, 2), "ki95_oben [%]": round(100 * _hi, 2)}))
_ = tabelle_speichern(df_z31, "05_31_negativkontrolle_r3")
df_neg3[df_neg3["ergebnis"] != "kein"].to_csv(
    ERR / "05_31_negativkontrolle_r3_faelle.csv", index=False, encoding="utf-8")   # privat

In [ ]:
# =============================================================================
# Zelle 32 – Negativkontrolle R3: Fehlvorschläge und mehrdeutiger Fall
# =============================================================================
# Zweck: Die 4 eindeutig fremden Treffer und den mehrdeutigen Fall ansehen,
#   um ein Veto abzuleiten (z. B. gemeinsame Titelwörter neben der Nummer).
# Legende: gesucht = Original (versteckt) | vorschlag = Treffer(s) mit gleicher
#   Nummer im selben Ordner.
# Datenschutz: nur lokale Anzeige.
# =============================================================================

_f32 = df_neg3[df_neg3["ergebnis"].isin(["eindeutig_fremd", "mehrdeutig"])].copy()
_f32["gesucht"] = _f32["pfad"].map(posixpath.basename)
# Lesbare Tabelle im Notebook (HTML, Zeilenumbruch in langen Zellen)
_f32["ordner"] = _f32["ordner"].str.replace("Musik/", "", regex=False)
# HTML-Tabelle ohne Zusatzpaket (jinja2 nicht installiert)
with pd.option_context("display.max_colwidth", None):
    display(_f32[["ergebnis", "ordner", "gesucht", "vorschlag"]].reset_index(drop=True))

In [ ]:
# =============================================================================
# Zelle 33 – Veto V3 für R3: Treffer = gesuchter Name + Anhang (Version) -> kein R3
# =============================================================================
# Zweck: Wirkung von V3 auf Negativkontrolle (4 Fehlvorschläge) und Gold (13
#   richtige R3-Treffer) messen.
# Regel V3: Stamm des Treffers beginnt mit dem Stamm des gesuchten Namens und
#   ist länger -> Version, R3 entscheidet nicht (Behandlung wie R2: Vorschlag).
# Erwartung: Negativkontrolle 4 -> 0 | Gold 13 -> 13.
# Legende: verworfen_v3 = Veto greift.
# =============================================================================

def veto_v3(gesucht: str, treffer: str) -> bool:
    g, _ = _stamm_endung(gesucht)
    t, _ = _stamm_endung(treffer)
    return len(t) > len(g) and t.startswith(g)


_neg = _f32[_f32["ergebnis"] == "eindeutig_fremd"]
_neg_v3 = [veto_v3(g, v) for g, v in zip(_neg["gesucht"], _neg["vorschlag"])]

_gold = pd.concat([
    df_r3.loc[df_r3["ergebnis"] == "eindeutig_richtig", ["pfad", "treffer"]],
    df_label_r3[["pfad", "treffer"]]])
_gold_v3 = [veto_v3(posixpath.basename(p), t) for p, t in zip(_gold["pfad"], _gold["treffer"])]

df_z33 = pd.DataFrame([
    {"population": "negativkontrolle_fremd", "faelle [Pfade]": len(_neg),
     "verworfen_v3 [Pfade]": sum(_neg_v3), "bleiben [Pfade]": len(_neg) - sum(_neg_v3)},
    {"population": "gold_richtig", "faelle [Pfade]": len(_gold),
     "verworfen_v3 [Pfade]": sum(_gold_v3), "bleiben [Pfade]": len(_gold) - sum(_gold_v3)}])
display(df_z33)
_ = tabelle_speichern(df_z33, "05_33_veto_v3")

### Entscheidung 21.11.2026 – Regel R3 "Tracknummer im Ordner" mit Veto V3

**Regel R3:** Zielordner = vorhandener Ordner, sonst Ordner mit höchstem Ordner-Jaccard (>= 0,5, eindeutig vorne). Kandidat = einzige Audiodatei mit gleicher führender Tracknummer.
**Veto V3:** Treffer beginnt mit dem gesuchten Namen und ist länger (Name + Anhang = Version) -> R3 entscheidet nicht, Behandlung wie R2.

| Messung | Ergebnis |
|---|---|
| Gold (Nutzerziele 2 + Nutzerprüfung 11) | 13/13 richtig, V3 verwirft 0 |
| Negativkontrolle (916 Pfade mit Tracknummer, 94 Ordner) | ohne V3: 4 fremd (0,44 %, KI 0,00–1,22 %), 1 mehrdeutig; mit V3: 0 fremd |
| Mehrdeutig im Sammelordner (Nummer = Set-Teil) | keine Aktion (gewollt) |

**Entscheidung:** R3 + V3 = **Vorschlag** mit Sammelentscheidung je Ordnerpaar (P13). Begründung: V3 an denselben Daten abgeleitet (keine unabhängige Prüfung), Gold aus nur 2 Ordnern, konservative Obergrenze über Ordner ca. 3,2 %. Hochstufung auf "sicher" möglich, wenn Nutzerentscheidungen aus weiteren Ordnern als Labels vorliegen.

### Stand der 62 ungelösten Fälle aus NB04 nach Phase-4-Regeln

| Ergebnis | Fälle | Regel |
|---|---|---|
| sicher | 6 | R1 (4), mehrfach_audiogleich (2) |
| Vorschlag | 19 | R2 (6), R3 + V3 (13) |
| album_beschaedigt | 11 | Präfix + 0-Byte-Ordner |
| fehlt | 26 | A 13, B 4, C 7, D 2: kein Regel-Kandidat; Nutzer oder Sicherung (Bonus) |

**Leitfrage:** Alle wirksamen Regeln sind deterministisch. Die verbleibenden 26 Fälle haben keinen Kandidaten auf dem Stick (umbenannt mit anderem Titel, verloren oder nie vorhanden); auch ML oder ein LLM kann hier nichts zuordnen, was nicht vorhanden ist.

In [ ]:
# =============================================================================
# Zelle 34 – Index mit SHA-256 (Manifest) und Audio-Hash (nur Namensdubletten)
# =============================================================================
# Zweck: Index für den neuen Resolver-Lauf (4.11) erzeugen: Spalten sha256 (aus
#   Manifest NB04 Z.08) und audio_sha256 (index.ergaenze_audio_hash, mp3 mit
#   mehrfach vorkommendem Namen). Laufzeit messen -> Abschätzung Vollauf (4.10).
# Kontrolle: src-Funktion = Notebook-Fassung (Zelle 27) für die 6 Gruppe-M-Dateien.
# Rein lesend (P1/P7-Ausnahme Hash). Stick muss eingesteckt sein.
# Ausgabe: data/interim/05_34_index_audiohash.csv (privat).
# =============================================================================

import time

from playlist_agent.index import ergaenze_audio_hash

STICK = projekt.stick(cfg)
df_ix34 = df_ix.merge(df_man[["pfad_relativ", "sha256"]], on="pfad_relativ", how="left")

_mp3 = (df_ix34["endung"] == ".mp3") & ~df_ix34["appledouble"]
_namen = df_ix34["pfad_relativ"].map(lambda p: p.rsplit("/", 1)[-1].lower())
_dup = _mp3 & (_namen.map(_namen[_mp3].value_counts()).fillna(0) >= 2)
print(f"mp3 gesamt: {_mp3.sum()} ({df_ix34.loc[_mp3, 'bytes'].sum() / 2**30:.1f} GiB) | "
      f"davon Namensdubletten: {_dup.sum()} ({df_ix34.loc[_dup, 'bytes'].sum() / 2**30:.2f} GiB)")

_t0 = time.perf_counter()
df_ix34 = ergaenze_audio_hash(df_ix34, STICK, nur_namensdubletten=True)
_dt = time.perf_counter() - _t0
_gib = df_ix34.loc[_dup, "bytes"].sum() / 2**30
print(f"Laufzeit: {_dt:.1f} s | Rate: {_gib / _dt * 1024:.0f} MiB/s" if _dt > 0 else "")
print(f"Abschätzung alle mp3: {df_ix34.loc[_mp3, 'bytes'].sum() / 2**30 / (_gib / _dt) / 60:.1f} min"
      if _gib > 0 else "")

# Kontrolle gegen Zelle 27
_ah_src = dict(zip(df_ix34["pfad_relativ"].map(lambda s: nfc(str(s))), df_ix34["audio_sha256"]))
_gleich = [_ah_src.get(nfc(r)) == h for r, h in zip(_rel, df_m24["audio_hash"])]
print(f"Kontrolle Zelle 27: {sum(_gleich)} von {len(_gleich)} Audio-Hashes gleich")

_pfad34 = INTERIM / "05_34_index_audiohash.csv"
if not _pfad34.exists():
    df_ix34.to_csv(_pfad34, index=False, encoding="utf-8")                    # privat
print(f"gespeichert/vorhanden: {_pfad34.name}, {df_ix34['audio_sha256'].notna().sum()} Audio-Hashes")

In [ ]:
# =============================================================================
# Zelle 35 – Neuer Resolver-Lauf (Phase-4-Regeln) und Regressionsprüfung gegen NB04
# =============================================================================
# Zweck: Alle Einträge (Crates: eindeutige Pfade; übrige: Einträge) mit dem neuen
#   Resolver auflösen, gleiche Vorstufen wie NB04 Z.05 (Stream, Linter).
# Index: Zelle 34 (sha256 aus Manifest, audio_sha256 für Namensdubletten);
#   leere Ordner vom Stick (index.leere_ordner); Regeln aus config.yaml.
# Regressionskriterien:
#   K1 NB04 sicher/vorschlag(name)/album_beschaedigt/ausgeschlossen/stream ->
#      Klasse, Stufe, Ziel unverändert (Erwartung: 0 Abweichungen)
#   K2 Geändert nur NB04 fehlt/mehrfach; auf Pfad-Ebene (62 Fälle) erwartet:
#      sicher 6 | vorschlag 19 | album_beschaedigt 11 | fehlt 26
# Datenschutz: Zählwerte in tables/; Ergebnisse privat in data/interim.
# =============================================================================

from playlist_agent.index import leere_ordner
from playlist_agent.linter import ist_stream, pruefe_eintrag
from playlist_agent.resolver import Resolver, regeln_aus_config

AUDIO = cfg["index"]["audio_endungen"]
REGELN = regeln_aus_config(cfg)
print("Regeln:", REGELN)
resolver_neu = Resolver(df_ix34, AUDIO, leere_ordner=leere_ordner(STICK), regeln=REGELN)


def bewerte_neu(pfad: str) -> dict:
    """Wie NB04 Z.05: Stream -> Linter -> Resolver."""
    if ist_stream(pfad):
        return {"klasse": "stream", "stufe": "webradio", "pfad_ziel": pfad.strip()}
    b = pruefe_eintrag(pfad, AUDIO)
    if b:
        return {"klasse": b.klasse, "stufe": b.untertyp, "pfad_ziel": None}
    e = resolver_neu.aufloesen(pfad)
    return {"klasse": e.klasse, "stufe": e.stufe, "pfad_ziel": e.pfad}


_neu_c = pd.DataFrame([bewerte_neu(p) for p in df_rc["eintrag"]]).add_suffix("_neu")
_neu_p = pd.DataFrame([bewerte_neu(k) for k in df_rp["kanonisch"].astype(str)]).add_suffix("_neu")
df_v_c = pd.concat([df_rc.reset_index(drop=True), _neu_c], axis=1).rename(columns={"eintrag": "pfad"})
df_v_p = pd.concat([df_rp.reset_index(drop=True), _neu_p], axis=1).rename(columns={"kanonisch": "pfad"})
df_v = pd.concat([df_v_c.assign(quelle="crates"), df_v_p], ignore_index=True)


def _gleich(a, b) -> bool:
    return (pd.isna(a) and pd.isna(b)) or a == b


df_v["geaendert"] = [not (_gleich(k, kn) and _gleich(s, sn) and _gleich(z, zn))
                     for k, kn, s, sn, z, zn in zip(df_v["klasse"], df_v["klasse_neu"], df_v["stufe"],
                                                     df_v["stufe_neu"], df_v["pfad_ziel"], df_v["pfad_ziel_neu"])]
df_v["alt_offen"] = (df_v["klasse"] == "fehlt") | (df_v["stufe"] == "mehrfach")

# K1
_k1 = df_v[~df_v["alt_offen"] & df_v["geaendert"]]
print(f"\nK1: Abweichungen bei NB04-Klassen außer fehlt/mehrfach: {len(_k1)} (Erwartung 0)")

# K2 auf Pfad-Ebene
_k2 = (df_v[df_v["alt_offen"]].drop_duplicates(["quelle", "pfad"])   # wie NB05: je Quelle
       .groupby("klasse_neu").size().rename("pfade [Pfade]"))
print(f"\nK2: NB04 fehlt/mehrfach, eindeutige Pfade: {_k2.sum()} (Erwartung 62)")
print(_k2.to_string())
print("\nNeue Stufen (eindeutige Pfade):")
print(df_v[df_v["alt_offen"]].drop_duplicates(["quelle", "pfad"])
      .groupby(["klasse_neu", "stufe_neu"]).size().to_string())

df_z35 = (df_v.groupby(["quelle", "klasse", "klasse_neu"]).size()
          .rename("anzahl [Eintraege]").reset_index())
_ = tabelle_speichern(df_z35, "05_35_resolver_alt_neu")
_pfad35 = INTERIM / "05_35_resolver_alt_neu.csv"
if not _pfad35.exists():
    df_v.to_csv(_pfad35, index=False, encoding="utf-8")                           # privat
if len(_k1):
    _k1.to_csv(ERR / "05_35_k1_abweichungen.csv", index=False, encoding="utf-8")  # privat

In [ ]:
# =============================================================================
# Zelle 36 – Kennzahlen alt/neu auf Eintrags-Ebene (übrige Quellen) + Spalten Crate-Einträge
# =============================================================================
# Zweck: Wirkung der Phase-4-Regeln dort messen, wo sie in Playlists ankommt
#   (Einträge). Übrige Quellen: 1 Zeile = 1 Eintrag. Crates: Spalten der
#   Eintragsdatei NB02 Z.02 feststellen (Zuordnung Eintrag -> Pfad, Zelle 37).
# Legende: alt = NB04 Baseline A | neu = Phase-4-Regeln | diff = neu - alt.
# Erwartung übrige Quellen: Summe 10.289; Änderungen nur aus fehlt (71) und mehrfach (2).
# =============================================================================

_a = df_v_p["klasse"].value_counts().rename("alt [Eintraege]")
_n = df_v_p["klasse_neu"].value_counts().rename("neu [Eintraege]")
df_z36 = pd.concat([_a, _n], axis=1).fillna(0).astype(int)
df_z36["diff [Eintraege]"] = df_z36["neu [Eintraege]"] - df_z36["alt [Eintraege]"]
df_z36.loc["Summe"] = df_z36.sum()
print("Übrige Quellen (Einträge):")
print(df_z36.to_string())
_ = tabelle_speichern(df_z36.reset_index(names="klasse"), "05_36_kennzahlen_uebrige_quellen")

_ce = pd.read_csv(INTERIM / "02_02_crate_eintraege.csv", nrows=5)
print("\nSpalten 02_02_crate_eintraege.csv:", list(_ce.columns))
print("Zeilen:", sum(1 for _ in open(INTERIM / "02_02_crate_eintraege.csv", encoding="utf-8")) - 1)

In [ ]:
# =============================================================================
# Zelle 37 – Crates (Einträge) und vollständige Playlists je Quelle, alt/neu
# =============================================================================
# Zweck: Kennzahlen für den Machbarkeitsbericht: Crate-Einträge je Klasse und
#   Anzahl vollständiger Playlists (alle Audio-Einträge geschrieben: sicher oder
#   stream; ausgeschlossen zählt nicht) vor/nach Phase 4.
# Zuordnung Crate-Eintrag -> Pfad: NB02 Z.02 (crate, position, pfad), NFC, '/'.
# Erwartung alt: Crates 7.915 Einträge (sicher 7.225 | vorschlag 168 | fehlt 13
#   | album_beschaedigt 509); übrige Quellen 185 Playlists, 57 vollständig.
# =============================================================================

df_ce = pd.read_csv(INTERIM / "02_02_crate_eintraege.csv")
df_ce["pfad"] = df_ce["pfad"].map(lambda s: nfc(str(s)).replace("\\", "/"))
_map = df_v_c.assign(pfad=df_v_c["pfad"].map(lambda s: nfc(str(s)).replace("\\", "/")))
# Zuordnung muss je normalisiertem Pfad eindeutig sein (NFC/NFD-Varianten fallen zusammen)
_dup = _map["pfad"].duplicated(keep=False)
_widerspruch = _map[_dup].groupby("pfad")[["klasse", "klasse_neu"]].nunique().gt(1).any(axis=1).sum()
print(f"Pfade in df_v_c: {len(_map)} | doppelte Schlüssel nach NFC: {_dup.sum()} Zeilen | "
      f"davon mit widersprüchlichem Ergebnis: {_widerspruch}")
assert _widerspruch == 0, "Doppelte Pfade mit verschiedenem Ergebnis – erst prüfen"
_map = _map.drop_duplicates("pfad")
df_ce = df_ce.merge(_map[["pfad", "klasse", "klasse_neu"]], on="pfad", how="left")
print(f"Crate-Einträge: {len(df_ce)} | ohne Zuordnung: {df_ce['klasse'].isna().sum()}")

_a = df_ce["klasse"].value_counts().rename("alt [Eintraege]")
_n = df_ce["klasse_neu"].value_counts().rename("neu [Eintraege]")
df_z37c = pd.concat([_a, _n], axis=1).fillna(0).astype(int)
df_z37c["diff [Eintraege]"] = df_z37c["neu [Eintraege]"] - df_z37c["alt [Eintraege]"]
df_z37c.loc["Summe"] = df_z37c.sum()
print("\nCrates (Einträge):")
print(df_z37c.to_string())


def vollstaendig(df: pd.DataFrame, schluessel: list[str], spalte: str) -> pd.Series:
    """Je Playlist: True, wenn alle Einträge außer 'ausgeschlossen' sicher/stream sind."""
    rel = df[df[spalte] != "ausgeschlossen"]
    ok = rel[spalte].isin(["sicher", "stream"]).groupby([rel[s] for s in schluessel]).all()
    return ok.reindex(df.groupby(schluessel).size().index, fill_value=False)   # nur ausgeschlossen -> False


_zeilen37 = []
for _q, _df, _key in (("crates", df_ce, ["crate"]),
                      ("uebrige_quellen", df_v_p, ["quelle", "datei"])):
    _alt, _neu = vollstaendig(_df, _key, "klasse"), vollstaendig(_df, _key, "klasse_neu")
    _zeilen37.append({"quelle": _q, "playlists [Playlists]": len(_alt),
                      "vollstaendig_alt [Playlists]": int(_alt.sum()),
                      "vollstaendig_neu [Playlists]": int(_neu.sum()),
                      "diff [Playlists]": int(_neu.sum() - _alt.sum())})
df_z37p = pd.DataFrame(_zeilen37)
print("\nVollständige Playlists:")
print(df_z37p.to_string(index=False))
_ = tabelle_speichern(df_z37c.reset_index(names="klasse"), "05_37_kennzahlen_crates")
_ = tabelle_speichern(df_z37p, "05_37_vollstaendige_playlists")

### Ergebnis 21.11.2026 – Neuer Resolver-Lauf (Arbeitspaket 4.11)

**Regression (Zelle 35):**
- K1: alle in NB04 bereits aufgelösten Einträge (sicher, Vorschlag name, album_beschaedigt, ausgeschlossen, stream) unverändert: 0 Abweichungen.
- K2: die 62 offenen Fälle aus NB04 (fehlt/mehrfach, je Quelle und Pfad) exakt wie in NB05 ermittelt: zeichen 4, mehrfach_audiogleich 2, praefix 6, tracknummer 13, 0-Byte praefix 8 / lose 3, fehlt 26.
- Kontrolle Audio-Hash: src-Funktion = Notebook-Fassung (6/6).

**Einträge alt (NB04) -> neu (Phase 4):**

| Klasse | Crates | Übrige Quellen | Summe diff |
|---|---|---|---|
| sicher | 7.225 -> 7.231 | 8.493 -> 8.498 | +11 |
| Vorschlag | 168 -> 172 | 1.026 -> 1.042 | +20 |
| album_beschaedigt | 509 -> 509 | 678 -> 693 | +15 |
| fehlt | 13 -> 3 | 71 -> 35 | -46 |

**Vollständige Playlists:** Crates 128 -> 130 (von 167), übrige Quellen 57 -> 58 (von 185).

**Befund Zuordnung:** 390 Crate-Pfade fallen nach NFC paarweise zusammen (NFC/NFD-Varianten), alle mit gleichem Ergebnis; Zuordnung Eintrag -> Pfad daher über eindeutigen NFC-Schlüssel.

**Leitfrage:** Phase 4 löst wenige, aber die schwierigsten Einträge, ausschliesslich mit deterministischen Regeln. Die grossen offenen Blöcke sind kein Erkennungsproblem: Datenverlust durch 0-Byte-Ordner (1.202 Einträge, nur aus Sicherung rückholbar) und Vorschläge, deren Identität belegt ist, deren Reparaturart aber der Nutzer entscheidet (1.214 Einträge, P13). Für beide bringen Fuzzy, ML oder LLM keinen Mehrwert; der Nutzen liegt in Bericht und Bedienung.

In [ ]:
# =============================================================================
# Zelle 38 – Audio-Hash aller mp3 und Dubletten-Messung (Arbeitspaket 4.10)
# =============================================================================
# Zweck: Hypothese prüfen: Kopien mit bearbeiteten Tags (audio-gleich, nicht
#   bitgleich) und namensgleiche Dateien mit verschiedener Musik (Versionen).
# Legende: bitgleich = SHA-256 ganze Datei gleich (Manifest) | audiogleich =
#   SHA-256 ohne ID3v2/APEv2/ID3v1 gleich | Gruppe = >= 2 Dateien gleichen
#   Inhalts | Paare = Summe n*(n-1)/2.
# Erwartung: bitgleiche mp3-Paare <= 45 (NB02: alle Formate); audiogleich >= bitgleich.
# Rein lesend (P1/P7-Ausnahme Hash). Laufzeit ca. 10 min (Zelle 34).
# Ausgabe: data/interim/05_38_audiohash_alle_mp3.csv (privat; vorhanden -> laden).
# =============================================================================

_pfad38 = INTERIM / "05_38_audiohash_alle_mp3.csv"
if _pfad38.exists():
    df_ah = pd.read_csv(_pfad38, low_memory=False)
    print(f"geladen: {_pfad38.name}")
else:
    _t0 = time.perf_counter()
    df_ah = ergaenze_audio_hash(df_ix34.drop(columns="audio_sha256"), STICK,
                                nur_namensdubletten=False)
    print(f"Laufzeit: {(time.perf_counter() - _t0) / 60:.1f} min")
    df_ah.to_csv(_pfad38, index=False, encoding="utf-8")                      # privat

_m = df_ah[df_ah["audio_sha256"].notna()].copy()
print(f"mp3 mit Audio-Hash: {len(_m)}")


def dubletten(spalte: str) -> dict:
    g = _m.groupby(spalte).size()
    g = g[g >= 2]
    return {"gruppen [Gruppen]": len(g), "dateien [Dateien]": int(g.sum()),
            "paare [Paare]": int((g * (g - 1) // 2).sum())}


df_z38 = pd.DataFrame([{"art": "bitgleich", **dubletten("sha256")},
                       {"art": "audiogleich", **dubletten("audio_sha256")}])
print("\nDubletten (mp3):")
print(df_z38.to_string(index=False))

# Namensgleiche Dateien: gleiche Musik oder verschiedene Versionen?
_m["name"] = _m["pfad_relativ"].map(lambda p: p.rsplit("/", 1)[-1].lower())
_ng = _m.groupby("name").agg(n=("audio_sha256", "size"),
                             audio=("audio_sha256", "nunique"),
                             bit=("sha256", "nunique"))
_ng = _ng[_ng["n"] >= 2]
_ng["art"] = np.select([_ng["bit"] == 1, _ng["audio"] == 1],
                       ["bitgleich", "nur Tags verschieden"], default="Musik verschieden")
df_z38n = _ng["art"].value_counts().rename("namensgruppen [Gruppen]").reset_index()
print(f"\nNamensgleiche mp3: {int(_ng['n'].sum())} Dateien in {len(_ng)} Namensgruppen")
print(df_z38n.to_string(index=False))

_ = tabelle_speichern(df_z38, "05_38_dubletten_mp3")
_ = tabelle_speichern(df_z38n, "05_38_namensgruppen_mp3")
_ng.reset_index().to_csv(ERR / "05_38_namensgruppen_faelle.csv", index=False, encoding="utf-8")  # privat

### Ergebnis 21.11.2026 – Audio-Hash aller mp3, Dubletten (Arbeitspaket 4.10)

Laufzeit 9,3 min für 4.924 mp3 (71,6 GiB; Abschätzung aus Zelle 34: 9,6 min).

| Art | Gruppen | Dateien | Paare |
|---|---|---|---|
| bitgleich (ganze Datei) | 45 | 90 | 45 |
| audiogleich (ohne Tags) | 80 | 166 | 92 |

- +47 Paare gleicher Musik, die sich nur in den Tags unterscheiden; der Audio-Hash findet doppelt so viele echte Kopien.
- Bitgleiche mp3-Paare = 45 = NB02 (alle Formate). Hypothese: keine bitgleichen wav/aif-Paare.

**Namensgleiche mp3 (469 Dateien, 109 Namensgruppen):** Musik verschieden 71 (65 %) | bitgleich 27 | nur Tags verschieden 11. Gleicher Name bedeutet in zwei von drei Fällen andere Musik (Versionen oder gleich benannte verschiedene Titel; nicht unterschieden).

**Folgen:** `mehrfach_audiogleich` greift nur bei Inhaltsgleichheit (38 Gruppen), sonst bleibt der Fall Vorschlag. Audio-gleiche Gruppen sind Kandidaten für einen späteren Aufräum-Bericht (nur Vorschlag, P1/P4). Option für den Index: Audio-Hash für alle mp3 statt nur Namensdubletten (Kosten ca. 10 min je Vollscan).

In [ ]:
# =============================================================================
# Zelle 39 – Integration belege.py: Hinweis je neuem Vorschlag (praefix/tracknummer)
# =============================================================================
# Zweck: src-Funktion belege.hinweis auf die neuen Vorschläge aus Zelle 35
#   anwenden; Abgleich mit Zelle 22 (R2-Gold).
# Erwartung (Zelle 22, R2-Gold 6): 3 Dauer+Grösse passt | 2 Dauer passt,
#   Grösse abweichend | 1 Dauer -0.60 s, Grösse abweichend.
# Legende: hinweis = Text, wie er im Bericht (Abschnitt 4) erscheint.
# Datenschutz: nur lokale Anzeige.
# =============================================================================

from playlist_agent.belege import hinweis, serato_meta

META_DB = serato_meta(df_db)
_nv = (df_v[df_v["stufe_neu"].isin(["praefix", "tracknummer"])]
       .drop_duplicates(["quelle", "pfad"]).copy())
_nv["hinweis"] = [hinweis(*META_DB.get(nfc(str(p)).replace("\\", "/"), (np.nan, np.nan)),
                          DAUER.get(z, np.nan), BYTES.get(z, np.nan))
                  for p, z in zip(_nv["pfad"], _nv["pfad_ziel_neu"])]
_nv["gesucht"] = _nv["pfad"].map(posixpath.basename)
print(_nv.groupby("stufe_neu")["hinweis"]
      .apply(lambda s: (s != "keine Serato-Daten").sum()).rename("mit Serato-Daten").to_string())
print()
print(_nv[["stufe_neu", "gesucht", "hinweis"]].sort_values(["stufe_neu", "gesucht"])
      .to_string(index=False))

## Abschluss Notebook 05 – Phase 4 Matching (21.11.2026)

**Ausgangslage:** 62 ungelöste Fälle aus NB04 (je Quelle und Pfad): fehlt 60, mehrfach 2.

**Regeln (alle deterministisch, in `src`, 186 Tests):**

| Regel | Klasse | Beleg |
|---|---|---|
| R1 Zeichen (Rand-Leerzeichen, Gross/Klein, 1 verlorenes Zeichen) | sicher | Gold 4/4; Negativkontrolle 0/2.505 |
| R2 Name + Anhang | Vorschlag | Gold 6/6; Negativkontrolle 26 Fehlvorschläge (1,04 %), Kaskade fängt 0 ab |
| R3 Tracknummer im Ordner + Veto V3 | Vorschlag | Gold 13/13; Negativkontrolle 4 -> 0 mit V3 (Veto an denselben Daten abgeleitet) |
| 0-Byte-Ordner mit Präfix / lose | album_beschaedigt | 11/11 vom Nutzer bestätigt |
| mehrfach_audiogleich (Audio-Hash ohne Tags) | sicher, Ort per Ordner-Jaccard | 2/2; Datei-Hash verschieden, Audio-Hash gleich |
| mehrfach_bitgleich, Ort per Ordner-Jaccard | sicher, Gleichstand -> Vorschlag | einheitlich mit audiogleich |
| Serato-Hinweis (Dauer/Grösse) | nur Hinweis im Bericht | trennt Version/Umbenennung nicht (Nutzer schneidet Titel) |

**Verworfen:** Ziffern-Veto V1 (1/26 abgefangen), Grössen-/Dauer-Toleranz als Beleg für "sicher", ML-Paar-Klassifikator für umbenannte Dateien (Merkmale trennen nicht).

**Wirkung (Einträge, Zellen 35–37):** fehlt 84 -> 38, sicher +11, Vorschlag +20, album_beschaedigt +15; vollständige Playlists Crates 128 -> 130, übrige Quellen 57 -> 58. Regression: 0 Änderungen an bereits Gelöstem.

**Audio-Hash (Zelle 38):** 92 audio-gleiche gegen 45 bitgleiche mp3-Paare; 71 von 109 Namensgruppen mit verschiedener Musik.

**Infrastruktur:** `projekt.py` (zentraler Kontext, `tabelle_speichern` ohne stilles Überschreiben, Stick über `anchor`), `belege.py`, `index.ergaenze_audio_hash`.

**Antwort auf die Leitfrage (Phase 4):** Für die Zuordnung reicht deterministisches Engineering mit Inhaltsbeweisen (Audio-Hash). Fuzzy-Ähnlichkeit ohne Vetos ist gefährlich (Nachbartitel), ML bringt bei fehlenden trennenden Merkmalen keinen Mehrwert. Die offenen Blöcke sind Datenverlust (0-Byte-Ordner, 1.202 Einträge) und Nutzerentscheidungen (1.214 Vorschläge) – Aufgaben für Bericht, UI und Sicherungs-Suche, nicht für Matching.

**Offen für spätere Phasen:** EXTINF-Dauer als Hinweis für M3U-Vorschläge; Hochstufung R3 bei weiteren Labels; Aufräum-Bericht audio-gleicher Kopien (nach Projektabschluss); Bewertungs-Playlists aus Tags (Bonus); ML-Mehrwert für die 1.214 Namens-Vorschläge (anderer Ordner) offen bewertet.